## SAX, Hierarchical clustering

Main pipeline. See README for details.

## 0. Configuration

Edit paths and headline modeling choices here. The rest of the notebook runs top-to-bottom.

In [1]:
from __future__ import annotations

import json
import os
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.signal import savgol_filter
from scipy.spatial.distance import squareform, pdist
from scipy.stats import norm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

from tslearn.piecewise import SymbolicAggregateApproximation, PiecewiseAggregateApproximation

# -----------------------------------------------------------------------------
# PATHS -- environment-variable overridable
# -----------------------------------------------------------------------------
# relative defaults so the
# notebook runs out of the box for any collaborator who clones the repo.
# Override via a .env file or shell export, e.g.:
#   export SAX_DATA_DIR=/path/to/data_events
#   export SAX_OUTPUT_DIR=/path/to/output/run_01
# DATA_DIR = Path(os.environ.get("SAX_DATA_DIR", "./data/data_events"))
# OUTPUT_DIR = Path(os.environ.get("SAX_OUTPUT_DIR", "./output/sax_run"))

# SP500_PATH = Path(os.environ.get("SAX_SP500_PATH", "./data/shock_detection/SP500_data.csv"))
# DJ_PATH = Path(os.environ.get("SAX_DJ_PATH", "./data/shock_detection/DOWJONES_data.csv"))
# NASDAQ_PATH = Path(os.environ.get("SAX_NASDAQ_PATH", "./data/shock_detection/NASDAQ100_data.csv"))
# RUSSELL_PATH = Path(os.environ.get("SAX_RUSSELL_PATH", "./data/shock_detection/RUSSELL2000_data.csv"))

DATA_DIR = Path(r"C:\Python\CSUREMM\data\processed_gt")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output")

STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed02*.csv"

START_DATE = "2022-01-01"
END_DATE = "2026-05-31"

# -----------------------------------------------------------------------------
# Data-specific filtering and preprocessing
# -----------------------------------------------------------------------------
MAX_ZERO_SHARE = 0.50
MAX_MISSING_SHARE = 0.02
MAX_INTERPOLATE_GAP = 7

DENOISE_WINDOW = 9
DENOISE_POLYORDER = 2
DETREND_WINDOW = 91

# -----------------------------------------------------------------------------
# SAX and hierarchical clustering
# -----------------------------------------------------------------------------
MINDIST_CHUNK_SIZE = 256
K_RANGE = range(2, 13)
RANDOM_STATE = 42
N_SEGMENTS = 96                   # see combined_wa_diagnostics.csv (Step 6): downstream
                                    # clustering quality is statistically flat across the
                                    # whole (w, a) grid
ALPHABET_SIZE = 9                  # same rationale as N_SEGMENTS above
FINAL_K = 10                       # primary k: see candidate_k_interpretation_summary.csv
CANDIDATE_K_VALUES = [7, 8, 9, 10, 11]  # k values fit and compared side-by-side in Steps 4-5

if FINAL_K not in CANDIDATE_K_VALUES:
    raise ValueError(
        f"FINAL_K={FINAL_K} must be one of CANDIDATE_K_VALUES={CANDIDATE_K_VALUES}. "
        "Add it to the list, or change FINAL_K to a value already being compared."
    )

# PAA BREAKPOINT_MODE = "empirical" derives breakpoints from the observed distribution
# instead of theoretical N(0,1) quantiles. Set back to "gaussian" to reproduce
# the original SAX behavior.
BREAKPOINT_MODE = "empirical"       # {"gaussian", "empirical"}

# This grid is for a diagnostic that reports how tight MINDIST is against true Euclidean distance
# at each (w, a) combination to justify the parameter selection
SEGMENT_GRID = [48, 64, 96, 128]
ALPHABET_GRID = [5, 7, 9, 11]
ABLATION_SAMPLE_SIZE = 150          # terms subsampled for the (w, a) tightness check

# Default to ward linkage, but empirically compare all four candidates
LINKAGE_METHOD = "ward"           # {"single", "average", "complete", "ward"}
LINKAGE_CANDIDATES = ["single", "average", "complete", "ward"]

# MINDIST produces exact/near-exact ties for genuinely distinct series
# (a direct consequence of quantization). Within TIE_EPS of each other, merge
# order is broken using the true Euclidean distance on the underlying series
# instead of being left to array/index order inside scipy's linkage.
TIE_EPS = 1e-6

# Step 5 also logs the stability/silhouette-recommended k (subject to this
# floor) alongside FINAL_K for comparison; it never overrides FINAL_K.
MIN_ACCEPTABLE_CLUSTER_SIZE = 15

# Stability: each iteration draws two overlapping subsamples and compares labels
STABILITY_SUBSAMPLES = 80
STABILITY_FRACTION = 0.80

# Cluster-index construction (used in Section 5)
CORE_STABILITY_QUANTILE = 0.75
MIN_CORE_TERMS = 10
TOP_N_CORE_TERMS = 10

SUBDIRS = [
    "00_provenance",
    "01_preprocessing",
    "02_sax",
    "03_distance",
    "04_clustering",
    "05_validation",
    "06_robustness",
    "07_visualization",
]

for sub in SUBDIRS:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

Output directory: C:\Python\CSUREMM\output


## 1. Load, filter, denoise, detrend, and normalize

In [2]:
def clean_term_from_folder(folder: Path) -> str:
    return folder.name.replace("_", " ").strip()


def find_value_column(df: pd.DataFrame) -> str:
    date_like = {"time", "date", "week"}
    candidates = [c for c in df.columns if c.lower() not in date_like]

    if not candidates:
        raise ValueError("No value column found.")

    numeric = [
        c for c in candidates
        if pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors="coerce"))
    ]

    return numeric[0] if numeric else candidates[0]


def max_consecutive_missing(s: pd.Series) -> int:
    missing = s.isna()

    if not missing.any():
        return 0

    groups = (missing != missing.shift()).cumsum()

    return int(
        missing
        .groupby(groups)
        .sum()
        .max()
    )


def load_one_series(folder: Path) -> pd.Series:
    stitched_dir = folder / STITCHED_SUBDIR
    files = sorted(stitched_dir.glob(STITCHED_GLOB)) if stitched_dir.exists() else []

    if not files:
        raise FileNotFoundError("missing stitched file")

    path = files[-1]
    df = pd.read_csv(path)

    date_col = next(
        (c for c in df.columns if c.lower() in {"time", "date", "week"}),
        df.columns[0],
    )

    value_col = find_value_column(df)

    dates = pd.to_datetime(df[date_col], errors="coerce")
    values = pd.to_numeric(df[value_col], errors="coerce")

    s = pd.Series(
        values.values,
        index=dates,
        name=clean_term_from_folder(folder),
    )

    s = s[~s.index.isna()]
    s = s[~s.index.duplicated(keep="last")]
    s = s.sort_index()

    s = s.loc[START_DATE:END_DATE]

    full_index = pd.date_range(
        START_DATE,
        END_DATE,
        freq="D",
    )

    return s.reindex(full_index)


def build_panel(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    series = {}

    folders = sorted([p for p in data_dir.iterdir() if p.is_dir()])

    expected_days = len(
        pd.date_range(
            START_DATE,
            END_DATE,
            freq="D",
        )
    )

    for folder in folders:
        term = clean_term_from_folder(folder)

        try:
            s = load_one_series(folder)

            observed_share = float(s.notna().mean())
            missing_share = float(s.isna().mean())
            zero_share = float((s.dropna() == 0).mean())
            longest_missing_gap = max_consecutive_missing(s)
            n_unique = int(s.nunique(dropna=True))

            reason = "kept"

            if zero_share > MAX_ZERO_SHARE:
                reason = "high_zero_share"
            elif missing_share > MAX_MISSING_SHARE:
                reason = "high_missing_share"
            elif longest_missing_gap > MAX_INTERPOLATE_GAP:
                reason = "long_missing_gap"
            elif n_unique <= 2:
                reason = "near_constant"

            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": int(s.notna().sum()),
                "observed_share": observed_share,
                "missing_share": missing_share,
                "zero_share": zero_share,
                "longest_missing_gap": longest_missing_gap,
                "n_unique": n_unique,
                "drop_reason": reason,
            })

            if reason == "kept":
                series[term] = s

        except Exception as e:
            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": 0,
                "observed_share": 0.0,
                "missing_share": 1.0,
                "zero_share": np.nan,
                "longest_missing_gap": np.nan,
                "n_unique": 0,
                "drop_reason": str(e),
            })

    panel = pd.DataFrame(series).sort_index()
    funnel = pd.DataFrame(rows).sort_values(["drop_reason", "term"])

    return panel, funnel


def interpolate_small_gaps(s: pd.Series) -> pd.Series:
    return (
        s.astype(float)
        .interpolate(
            method="time",
            limit=MAX_INTERPOLATE_GAP,
            limit_direction="both",
        )
    )


def denoise_series(s: pd.Series) -> pd.Series:
    x = s.astype(float)
    valid = x.dropna()

    if len(valid) < DENOISE_WINDOW or DENOISE_WINDOW % 2 == 0:
        return x

    filled = x.interpolate(
        method="time",
        limit_direction="both",
    )

    return pd.Series(
        savgol_filter(
            filled,
            DENOISE_WINDOW,
            DENOISE_POLYORDER,
        ),
        index=x.index,
        name=x.name,
    )


def detrend_series(s: pd.Series) -> pd.Series:
    trend = s.rolling(
        DETREND_WINDOW,
        center=True,
        min_periods=max(14, DETREND_WINDOW // 4),
    ).median()

    return s - trend


def robust_zscore(
    s: pd.Series,
    mad_floor_frac: float = 0.05,
    mad_ratio_threshold: float = 0.02,
    clip_mad_multiple: float = 4.0,
) -> pd.Series:
    x = s.astype(float)

    med = x.median(skipna=True)
    mad = (x - med).abs().median(skipna=True)

    if pd.isna(mad) or mad == 0:
        std = x.std(skipna=True)

        if pd.isna(std) or std == 0:
            return pd.Series(
                np.zeros(len(x)),
                index=x.index,
                name=x.name,
            )

        return (
            (x - x.mean(skipna=True)) / std
        ).rename(x.name)

    p05, p95 = x.quantile(0.05), x.quantile(0.95)
    wide_spread = (p95 - p05) / 3.29

    mad_ratio = (
        mad / wide_spread
        if wide_spread > 0
        else 1.0
    )

    is_degenerate = mad_ratio < mad_ratio_threshold

    if not is_degenerate:
        return (
            0.6745 * (x - med) / mad
        ).rename(x.name)

    mad_floored = (
        max(mad, mad_floor_frac * wide_spread)
        if wide_spread > 0
        else mad
    )

    z = (
        0.6745 * (x - med) / mad_floored
    ).rename(x.name)

    return z.clip(
        lower=-clip_mad_multiple,
        upper=clip_mad_multiple,
    )


def preprocess_panel(
    panel: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:

    stages = {}

    stages["filled"] = panel.apply(
        interpolate_small_gaps,
        axis=0,
    )

    stages["denoised"] = stages["filled"].apply(
        denoise_series,
        axis=0,
    )

    stages["detrended"] = stages["denoised"].apply(
        detrend_series,
        axis=0,
    )

    stages["normalized"] = stages["detrended"].apply(
        robust_zscore,
        axis=0,
    )

    return stages["normalized"], stages


def save_panel_stages(panel_raw: pd.DataFrame, stages: dict[str, pd.DataFrame]) -> None:
    panel_raw.to_csv(OUTPUT_DIR / "01_preprocessing" / "panel_raw_retained.csv")
    for name, df in stages.items():
        df.to_csv(OUTPUT_DIR / "01_preprocessing" / f"panel_{name}.csv")

In [3]:
panel_raw, filtering_funnel = build_panel(DATA_DIR)
filtering_funnel.to_csv(OUTPUT_DIR / "00_provenance" / "filtering_funnel.csv", index=False)

panel_norm, stages = preprocess_panel(panel_raw)
panel_norm = panel_norm.dropna(axis=1, how="all")
panel_norm = panel_norm.loc[:, panel_norm.notna().mean() > 0.95]
stages["normalized"] = panel_norm

save_panel_stages(panel_raw, stages)

summary = pd.DataFrame({
    "n_days": [panel_norm.shape[0]],
    "n_terms_retained": [panel_norm.shape[1]],
    "start_date": [panel_norm.index.min()],
    "end_date": [panel_norm.index.max()],
})
summary.to_csv(OUTPUT_DIR / "00_provenance" / "analysis_sample_summary.csv", index=False)

print(summary.to_string(index=False))
print("\nFiltering funnel:")
print(filtering_funnel["drop_reason"].value_counts(dropna=False).to_string())

 n_days  n_terms_retained start_date   end_date
   1612               847 2022-01-01 2026-05-31

Filtering funnel:
drop_reason
kept                  847
high_zero_share       352
high_missing_share      4


In [4]:
panel_norm.isna().sum().sum() == 0

np.True_

## 2. `tslearn` SAX representation

Each retained term is represented as one SAX string. The notebook uses `tslearn.piecewise.SymbolicAggregateApproximation`. Linearly fills missing values, only for the representation step, though previous steps should have already produced a clean panel without NAs.

In [5]:
def panel_to_tslearn_array(panel: pd.DataFrame) -> np.ndarray:
    filled = panel.astype(float).interpolate("time").ffill().bfill()
    X = filled.T.to_numpy(dtype=float)
    return X[:, :, None]


def sax_to_strings(codes_2d: np.ndarray, alphabet_size: int, index) -> pd.Series:
    alphabet = np.array(list("abcdefghijklmnopqrstuvwxyz"[:alphabet_size]))
    strings = ["".join(alphabet[row]) for row in codes_2d]
    return pd.Series(strings, index=index, name="sax_string")


def sax_breakpoints(alphabet_size: int) -> np.ndarray:
    """Gaussian breakpoints used by standard SAX (theoretical N(0,1) assumption)."""
    return norm.ppf(np.arange(1, alphabet_size) / alphabet_size)


def empirical_breakpoints(paa_values: np.ndarray, alphabet_size: int) -> np.ndarray:
    """
    Data-driven analogue of sax_breakpoints(): equiprobable quantiles of the
    pooled, observed PAA-coefficient distribution rather than theoretical
    N(0,1) quantiles. Use when the Gaussian assumption doesn't fit the data
    (see the fit diagnostic immediately below).
    """
    qs = np.arange(1, alphabet_size) / alphabet_size
    return np.quantile(paa_values, qs)


def discretize_paa(paa_values_2d: np.ndarray, breakpoints: np.ndarray) -> np.ndarray:
    """Assign each PAA coefficient to a symbol index given a set of breakpoints."""
    return np.digitize(paa_values_2d, breakpoints)


# throwing away extreme values to avoid distortion in SAX representation
panel_sax = panel_norm.clip(-5, 5)
X_ts = panel_to_tslearn_array(panel_sax)

paa = PiecewiseAggregateApproximation(n_segments=N_SEGMENTS)
paa_coefs = paa.fit_transform(X_ts)[:, :, 0]          # (n_terms, n_segments)
paa_flat = paa_coefs.ravel()

# BREAKPOINT_MODE (config, Section 0) selects Gaussian vs. empirical breakpoints.
if BREAKPOINT_MODE == "empirical":
    active_breakpoints = empirical_breakpoints(paa_flat, ALPHABET_SIZE)
else:
    active_breakpoints = sax_breakpoints(ALPHABET_SIZE)

sax_code_array = discretize_paa(paa_coefs, active_breakpoints)   # (n_terms, n_segments)
sax_codes = sax_code_array[:, :, None]                            # tslearn-style shape

sax_strings = sax_to_strings(sax_code_array, ALPHABET_SIZE, panel_norm.columns)

sax_df = pd.DataFrame({
    "term": sax_strings.index,
    "sax_string": sax_strings.values,
    "n_segments": N_SEGMENTS,
    "alphabet_size": ALPHABET_SIZE,
    "breakpoint_mode": BREAKPOINT_MODE,
})
sax_df.to_csv(OUTPUT_DIR / "02_sax" / "sax_strings.csv", index=False)

symbol_counts = (
    sax_df.assign(symbol=sax_df["sax_string"].map(list))
    .explode("symbol")
    .groupby(["term", "symbol"])
    .size()
    .unstack(fill_value=0)
)
symbol_counts.to_csv(OUTPUT_DIR / "02_sax" / "sax_symbol_counts.csv")

print("SAX strings:", sax_df.shape, " | breakpoint_mode:", BREAKPOINT_MODE)
sax_df.head()

SAX strings: (847, 5)  | breakpoint_mode: empirical


,term,sax_string,n_segments,alphabet_size,breakpoint_mode
0,2020 election map,gfadaebheagifafbebhicfbbaggbggcbfcbaiiaedbiadb...,96,9,empirical
1,2020 election results,fgbgfcchieihahhcaagihhcbeefehdedecbbiidaahiahf...,96,9,empirical
2,2022 election,bcgihaaiifbaaiiceadigeeeeeeeedgchfbdhdbcghidca...,96,9,empirical
3,2022 election results,edehgfchihgcaiieecdigeeeeeeeedfdfebfgecdfeibbb...,96,9,empirical
4,2024 election,gdcfcbeedahhhbccdbfihbabghiahgggheaagigaaiiaaa...,96,9,empirical


### Justify clipping value selection at $\pm 5 \text{ SD}$ is the right approach

In [6]:
for c in [2,3,4,5]:
    pct = np.mean(np.abs(panel_norm.to_numpy()) > c)
    print(c, pct)

2 0.18607492214530338
3 0.1243272856176082
4 0.09299351674718244
5 0.07572705886488877


In [7]:
# --- Diagnostic: does the Gaussian breakpoint assumption actually fit? ---
# Excess kurtosis > 0 or a small KS p-value indicates the pooled coefficients are more peaked / non-Gaussian
# which motivates BREAKPOINT_MODE = "empirical" above.
from scipy import stats as _stats

standardized = (paa_flat - paa_flat.mean()) / paa_flat.std()
excess_kurtosis = _stats.kurtosis(paa_flat, fisher=True)   # 0 for a true Gaussian
ks_stat, ks_p = _stats.kstest(standardized, "norm")

breakpoint_fit = pd.DataFrame({
    "metric": ["excess_kurtosis", "ks_statistic", "ks_pvalue", "n_paa_coefficients", "breakpoint_mode_used"],
    "value": [excess_kurtosis, ks_stat, ks_p, len(paa_flat), BREAKPOINT_MODE],
})
breakpoint_fit.to_csv(OUTPUT_DIR / "06_robustness" / "paa_gaussian_fit_diagnostic.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(paa_flat, bins=60, density=True, alpha=0.7, label="pooled PAA coefficients")
xs = np.linspace(paa_flat.min(), paa_flat.max(), 200)
axes[0].plot(xs, norm.pdf(xs, paa_flat.mean(), paa_flat.std()), lw=2, label="fitted Gaussian")
axes[0].set_title("Pooled PAA coefficients vs Gaussian fit")
axes[0].legend(frameon=False)
_stats.probplot(paa_flat, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot vs Gaussian")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_robustness" / "paa_gaussian_fit_diagnostic.png", dpi=200)
plt.close(fig)

print(f"Excess kurtosis: {excess_kurtosis:.3f}  (0 = Gaussian; > 0 = more peaked than Gaussian)")
print(f"KS test vs N(0,1): stat={ks_stat:.4f}, p={ks_p:.2e}")
print(f"-> breakpoint_mode currently in use: {BREAKPOINT_MODE}")

Excess kurtosis: 3.887  (0 = Gaussian; > 0 = more peaked than Gaussian)
KS test vs N(0,1): stat=0.1457, p=0.00e+00
-> breakpoint_mode currently in use: empirical


## 3. MINDIST

The clustering distance is SAX **MINDIST**, computed from the (Gaussian or
empirical, per `BREAKPOINT_MODE`) breakpoints from Section 2.

Before committing to `N_SEGMENTS` / `ALPHABET_SIZE`, a small ablation below
reports how tight MINDIST is against the true Euclidean distance across a grid
of `(n_segments, alphabet_size)` choices -- this is a diagnostic, not an
automatic override, so the hardcoded values above can be checked against
evidence.

Linkage defaults to **ward linkage**: it is the only merge rule for which
Kleinberg-style scale invariance + richness + consistency are jointly
satisfiable once stated over the dendrogram (Zadeh & Ben-David, 2009), which
is the property MINDIST-based clustering is implicitly leaning on. All four
standard linkage methods are compared empirically in Section 4 regardless, so
this default is a starting point, not an assumption baked in silently.

MINDIST also produces exact/near-exact distance ties for genuinely distinct
series (a direct consequence of quantization); these are broken using the true
Euclidean distance on the underlying series rather than left to array order.

In [8]:
def sax_symbol_distance_table(alphabet_size: int, breakpoints: np.ndarray | None = None) -> np.ndarray:
    """
    Pairwise SAX symbol distances used in MINDIST.

    Adjacent symbols have distance 0 because their intervals touch. Non-adjacent
    symbols are separated by the gap between the relevant breakpoints. Accepts an
    explicit breakpoints array so the table reflects whichever BREAKPOINT_MODE
    (Gaussian or empirical) was actually used to build the SAX codes.
    """
    bp = sax_breakpoints(alphabet_size) if breakpoints is None else breakpoints
    table = np.zeros((alphabet_size, alphabet_size), dtype=float)

    for i in range(alphabet_size):
        for j in range(alphabet_size):
            if abs(i - j) <= 1:
                table[i, j] = 0.0
            else:
                lo, hi = sorted((i, j))
                table[i, j] = bp[hi - 1] - bp[lo]
    return table

# NOTE: the (w, a) MINDIST-tightness ablation that used to run here has moved
# to Section 06 (Validity checks), alongside the reconstruction-elbow and
# downstream-quality checks it's meant to be read together with.

In [9]:
def sax_mindist_matrix(
    sax_codes: np.ndarray,
    terms: list[str],
    n_timestamps: int,
    n_segments: int,
    alphabet_size: int,
    symbol_dist: np.ndarray,
    chunk_size: int = MINDIST_CHUNK_SIZE,
) -> pd.DataFrame:
    """
    Compute SAX MINDIST between all term-level SAX strings.

    Parameters
    ----------
    sax_codes:
        Symbol-code array, shape (n_terms, n_segments, 1).
    terms:
        Term names in the same order as sax_codes.
    n_timestamps:
        Original equal-length series length before SAX compression.
    n_segments:
        Number of SAX/PAA segments.
    alphabet_size:
        Number of SAX symbols.
    symbol_dist:
        Precomputed symbol-to-symbol distance table (reflects the breakpoints
        actually used -- Gaussian or empirical -- rather than assuming Gaussian).
    chunk_size:
        Row chunk size to keep memory use stable.

    Returns
    -------
    pd.DataFrame
        Symmetric MINDIST matrix indexed and columned by term.
    """
    codes = np.asarray(sax_codes[:, :, 0], dtype=np.int16)
    n_terms, actual_segments = codes.shape

    if actual_segments != n_segments:
        raise ValueError(f"Expected {n_segments} SAX segments, got {actual_segments}.")
    if len(terms) != n_terms:
        raise ValueError("Number of terms does not match number of SAX code rows.")

    scale = np.sqrt(n_timestamps / n_segments)
    D = np.zeros((n_terms, n_terms), dtype=np.float32)

    for start in range(0, n_terms, chunk_size):
        stop = min(start + chunk_size, n_terms)
        block = codes[start:stop]
        per_segment = symbol_dist[block[:, None, :], codes[None, :, :]]
        D[start:stop, :] = scale * np.sqrt((per_segment ** 2).sum(axis=2))

    D = 0.5 * (D + D.T)
    np.fill_diagonal(D, 0.0)
    return pd.DataFrame(D, index=terms, columns=terms)


def true_euclidean_distance_matrix(panel: pd.DataFrame) -> pd.DataFrame:
    """True (full-resolution) Euclidean distance, used only to break MINDIST ties."""
    X = panel.T.to_numpy(dtype=float)
    D = squareform(pdist(X, metric="euclidean"))
    return pd.DataFrame(D, index=panel.columns, columns=panel.columns)


def break_mindist_ties(mindist_df: pd.DataFrame, euclid_df: pd.DataFrame, eps: float = TIE_EPS) -> pd.DataFrame:
    """
    MINDIST is a many-to-one lower bound and produces exact/near-exact ties for
    genuinely distinct series (see the duplicate-value counts in the SAX
    characterization diagnostics). Within `eps` those ties are effectively
    arbitrary and would otherwise be resolved by array/index order inside
    scipy's linkage. This adds a rank-preserving nudge from the true Euclidean
    distance, strictly smaller than `eps`, so no non-tied comparison is
    reordered but tied comparisons are resolved using real signal.
    """
    M = mindist_df.to_numpy(dtype=float)
    E = euclid_df.reindex(index=mindist_df.index, columns=mindist_df.columns).to_numpy(dtype=float)

    e_rank = E / (E.max() + 1e-9)          # in [0, 1), preserves Euclidean order
    nudge = eps * e_rank                    # strictly smaller than eps
    M_broken = M + nudge
    np.fill_diagonal(M_broken, 0.0)
    M_broken = 0.5 * (M_broken + M_broken.T)
    return pd.DataFrame(M_broken, index=mindist_df.index, columns=mindist_df.columns)


def hierarchical_labels(distance_df: pd.DataFrame, k: int, method: str | None = None) -> pd.Series:
    method = method or LINKAGE_METHOD
    condensed = squareform(distance_df.to_numpy(dtype=float), checks=False)
    Z = linkage(condensed, method=method)
    labels = fcluster(Z, t=k, criterion="maxclust")
    return pd.Series(labels, index=distance_df.index, name="cluster")


symbol_dist_table = sax_symbol_distance_table(ALPHABET_SIZE, active_breakpoints)

mindist = sax_mindist_matrix(
    sax_codes=sax_codes,
    terms=sax_strings.index.to_list(),
    n_timestamps=panel_norm.shape[0],
    n_segments=N_SEGMENTS,
    alphabet_size=ALPHABET_SIZE,
    symbol_dist=symbol_dist_table,
)
mindist.to_csv(OUTPUT_DIR / "03_distance" / "sax_mindist_matrix.csv")

mindist_summary = pd.DataFrame({
    "metric": ["n_terms", "n_segments", "alphabet_size", "min_offdiag", "median_offdiag", "mean_offdiag", "max_offdiag"],
    "value": [
        mindist.shape[0],
        N_SEGMENTS,
        ALPHABET_SIZE,
        np.nanmin(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmedian(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmean(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmax(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
    ],
})
mindist_summary.to_csv(OUTPUT_DIR / "03_distance" / "sax_mindist_summary.csv", index=False)

n_exact_ties = int(np.isclose(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy(), 0.0, atol=1e-9).sum())
print(f"Exact-zero off-diagonal MINDIST pairs before tie-breaking: {n_exact_ties}")

euclid_true = true_euclidean_distance_matrix(panel_sax)
mindist_tiebroken = break_mindist_ties(mindist, euclid_true, eps=TIE_EPS)
mindist_tiebroken.to_csv(OUTPUT_DIR / "03_distance" / "sax_mindist_matrix_tiebroken.csv")

mindist_summary

Exact-zero off-diagonal MINDIST pairs before tie-breaking: 16


,metric,value
0,n_terms,847.000000
1,n_segments,96.000000
2,alphabet_size,9.000000
3,min_offdiag,0.000000
4,median_offdiag,32.555050
5,mean_offdiag,31.949610
6,max_offdiag,54.985077


### Some diagnosis on SAX characterization

In [10]:
print("panel_norm shape:", panel_norm.shape)
print("N_SEGMENTS:", N_SEGMENTS)
print("ALPHABET_SIZE:", ALPHABET_SIZE)
print("FINAL_K:", FINAL_K)
print("NaNs:", panel_norm.isna().sum().sum())
print("Constant cols:", (panel_norm.std(axis=0) == 0).sum())

panel_norm shape: (1612, 847)
N_SEGMENTS: 96
ALPHABET_SIZE: 9
FINAL_K: 10
NaNs: 0
Constant cols: 0


In [11]:
codes = np.asarray(sax_codes[:, :, 0], dtype=int)

# 1. How many unique SAX words?
sax_words = ["".join(map(str, row)) for row in codes]
print("unique SAX words:", pd.Series(sax_words).nunique())
print(pd.Series(sax_words).value_counts().head(20))

# 2. Symbol usage
symbol_counts = pd.Series(codes.ravel()).value_counts().sort_index()
print(symbol_counts)

# 3. Per-term number of unique symbols
unique_symbols_per_term = pd.Series(
    [len(set(row)) for row in codes],
    index=sax_strings.index,
    name="n_unique_symbols"
)
print(unique_symbols_per_term.describe())
print(unique_symbols_per_term.value_counts().sort_index())

# 4. Distance distribution
offdiag = mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy().ravel()
offdiag = offdiag[~np.isnan(offdiag)]
print(pd.Series(offdiag).describe())
print(pd.Series(np.round(offdiag, 6)).value_counts().head(20))

unique SAX words: 843
434765278762088442386444444443535415642354811188478000377058100188803575828583466874443885008812    2
632521443077712231587101678076667400686008800080388000660588080367444444443434633752236744215743    2
007810088210088450187644445453687413724174871065786422671777170367444444444773573721133867203748    2
136206577531215576663704766516773321165766617047465167652112676457150565643575512145666562103746    2
650304174068505141782511066166215210880431803187087200430788171367444444444444444645545786106851    1
561652278487077200687721445473434211883007807577186211371688080178444444444543644881025772808880    1
126870088510088240386444444443627513731267832044376331141477011688410444424162266880147786008850    1
414621352177723140387031588066777701686008800186088101170688080367444444443434634862245743116861    1
047208788641226786770701777316884312277767706008662276552223687357150474625564422346765561105644    1
784164464444467676500707834543754434776884083218844016511047

In [12]:
for k in range(2, 11):
    labels = hierarchical_labels(mindist_tiebroken, k)

    print(
        k,
        labels.value_counts().values
    )

2 [671 176]
3 [613 176  58]
4 [461 176 152  58]
5 [436 176 152  58  25]
6 [436 152  90  86  58  25]
7 [436 152  90  58  46  40  25]
8 [436  90  80  72  58  46  40  25]
9 [301 135  90  80  72  58  46  40  25]
10 [301  90  88  80  72  58  47  46  40  25]


## 4. Hierarchical clustering

Fit one linkage tree from `mindist_tiebroken`, then cut it at every k in
`CANDIDATE_K_VALUES` -- cutting a fixed tree at different k is essentially
free, since scipy's `linkage()` (the expensive step) doesn't depend on k at
all. Each candidate's term assignments, cluster sizes, and dendrogram are
saved to their own `04_clustering/k_{k}/` folder, plus a side-by-side
`candidate_k_comparison.csv` for comparing structural stats at a glance.

`FINAL_K` (Step 0) selects which candidate becomes the notebook-wide primary
solution: `labels_final`, used by every downstream section (validation,
robustness, poster). Change `FINAL_K` and re-run to promote a different
candidate -- no need to touch `CANDIDATE_K_VALUES` or re-fit anything.

In [13]:
import shutil

CLUSTER_DIR = OUTPUT_DIR / "04_clustering"

if not isinstance(mindist_tiebroken, pd.DataFrame):
    raise TypeError("mindist_tiebroken must be a pandas DataFrame.")
if mindist_tiebroken.shape[0] != mindist_tiebroken.shape[1]:
    raise ValueError("mindist_tiebroken must be square.")
if not mindist_tiebroken.index.equals(mindist_tiebroken.columns):
    raise ValueError("Distance-matrix rows and columns must match in the same order.")

D_final = mindist_tiebroken.to_numpy(dtype=float)
if not np.all(np.isfinite(D_final)):
    raise ValueError("mindist_tiebroken contains NaN or infinite values.")
if not np.allclose(D_final, D_final.T, atol=1e-10):
    raise ValueError("mindist_tiebroken must be symmetric.")
if not np.allclose(np.diag(D_final), 0.0, atol=1e-10):
    raise ValueError("The diagonal of mindist_tiebroken must be zero.")

for k in CANDIDATE_K_VALUES:
    if not 2 <= k < len(mindist_tiebroken):
        raise ValueError(
            f"Every value in CANDIDATE_K_VALUES must be between 2 and "
            f"{len(mindist_tiebroken) - 1}; got k={k}."
        )

# One linkage tree for the whole notebook -- every candidate k below is just
# a different cut of this same tree, not a separate fit.
Z_SHARED = linkage(squareform(D_final, checks=False), method=LINKAGE_METHOD)


def fit_hierarchical_solution(k: int, save_dir: Path, show: bool = False) -> dict:
    """
    Cut the shared linkage tree at k, save its term assignments, cluster
    sizes, and dendrogram under `save_dir`, and return everything needed to
    interpret or compare this candidate later.
    """
    save_dir.mkdir(parents=True, exist_ok=True)

    labels = pd.Series(
        fcluster(Z_SHARED, t=k, criterion="maxclust"),
        index=mindist_tiebroken.index,
        name="cluster",
    ).astype(int)

    assignments = (
        labels.rename_axis("term").reset_index().sort_values(["cluster", "term"])
    )
    assignments.to_csv(save_dir / "cluster_assignments.csv", index=False)

    sizes = (
        labels.value_counts().sort_index()
        .rename_axis("cluster").reset_index(name="n_terms")
    )
    sizes["share_of_terms"] = sizes["n_terms"] / len(labels)
    sizes.to_csv(save_dir / "cluster_sizes.csv", index=False)

    color_threshold = Z_SHARED[-(k - 1), 2] if k > 1 else None
    fig, ax = plt.subplots(figsize=(14, 6))
    dendrogram(
        Z_SHARED, no_labels=True, color_threshold=color_threshold,
        above_threshold_color="gray", ax=ax,
    )
    ax.set_title(f"Hierarchical clustering dendrogram, k = {k}, linkage = {LINKAGE_METHOD}")
    ax.set_xlabel("Search terms")
    ax.set_ylabel("SAX MINDIST (tie-broken)")
    fig.tight_layout()
    fig.savefig(save_dir / f"dendrogram_k{k}.png", dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)

    return {"k": k, "labels": labels, "sizes": sizes}


CLUSTERING_BY_K = {
    k: fit_hierarchical_solution(k, CLUSTER_DIR / f"k_{k}", show=(k == FINAL_K))
    for k in CANDIDATE_K_VALUES
}

candidate_k_comparison = pd.DataFrame([
    {
        "k": k,
        "n_terms": len(result["labels"]),
        "min_cluster_size": int(result["sizes"]["n_terms"].min()),
        "max_cluster_size": int(result["sizes"]["n_terms"].max()),
        "median_cluster_size": float(result["sizes"]["n_terms"].median()),
        "largest_cluster_share": float(result["sizes"]["share_of_terms"].max()),
        "is_final_k": k == FINAL_K,
    }
    for k, result in CLUSTERING_BY_K.items()
])
candidate_k_comparison.to_csv(CLUSTER_DIR / "candidate_k_comparison.csv", index=False)

# Promote the configured FINAL_K to the single, notebook-wide "primary"
# solution that every downstream section (Steps 5-7) reads from here on.
labels_final = CLUSTERING_BY_K[FINAL_K]["labels"]
cluster_sizes_final = CLUSTERING_BY_K[FINAL_K]["sizes"]

# Mirror the primary solution under flat, k-independent filenames too, purely
# for discoverability -- its authoritative copy still lives in k_{FINAL_K}/.
(
    labels_final.rename_axis("term").reset_index().sort_values(["cluster", "term"])
).to_csv(CLUSTER_DIR / "final_cluster_assignments.csv", index=False)
cluster_sizes_final.to_csv(CLUSTER_DIR / "cluster_sizes_final.csv", index=False)
shutil.copy2(
    CLUSTER_DIR / f"k_{FINAL_K}" / f"dendrogram_k{FINAL_K}.png",
    CLUSTER_DIR / f"dendrogram_k{FINAL_K}.png",
)

print(f"Fit {len(CANDIDATE_K_VALUES)} candidate solutions: {CANDIDATE_K_VALUES}")
print(f"Primary (FINAL_K={FINAL_K}): {len(labels_final):,} terms across {FINAL_K} clusters.\n")
print("Candidate comparison (structural stats only -- see Step 5 for silhouette/stability):")
print(candidate_k_comparison.to_string(index=False))

Fit 5 candidate solutions: [7, 8, 9, 10, 11]
Primary (FINAL_K=10): 847 terms across 10 clusters.

Candidate comparison (structural stats only -- see Step 5 for silhouette/stability):
 k  n_terms  min_cluster_size  max_cluster_size  median_cluster_size  largest_cluster_share  is_final_k
 7      847                25               436                 58.0               0.514758       False
 8      847                25               436                 65.0               0.514758       False
 9      847                25               301                 72.0               0.355372       False
10      847                25               301                 65.0               0.355372        True
11      847                25               231                 70.0               0.272727       False


## 5. Cluster validation and interpretation

Two independent checks, both spanning multiple k:

1. **Statistical validation across the full `K_RANGE` (2-12)** -- silhouette
   and subsample stability for every k in that range, regardless of which
   ones were fit as full candidates in Step 4. Flags whether the
   stability/silhouette-recommended k agrees with `FINAL_K`, without ever
   overwriting it.
2. **Interpretive comparison across `CANDIDATE_K_VALUES`** -- term stability,
   core terms, and a cluster summary (including each cluster's MINDIST
   margin) for every candidate fit in Step 4, reusing those labels rather
   than recomputing them. `FINAL_K`'s interpretation is promoted as the
   notebook-wide primary summary that Steps 6-7 read from.

In [14]:
VALIDATION_DIR = OUTPUT_DIR / "05_validation"

def labels_from_submatrix(
    full_distance: pd.DataFrame,
    terms: list[str],
    k: int,
    method: str | None = None,
) -> pd.Series:
    method = method or LINKAGE_METHOD
    sub_distance = full_distance.loc[terms, terms]
    return hierarchical_labels(sub_distance, k=k, method=method)


def stability_for_k(
    full_distance: pd.DataFrame,
    k: int,
    n_subsamples: int,
    fraction: float,
    seed: int,
    method: str | None = None,
) -> pd.DataFrame:
    method = method or LINKAGE_METHOD
    rng = np.random.default_rng(seed)
    terms = np.asarray(full_distance.index)
    n_take = min(len(terms), max(k + 2, int(round(len(terms) * fraction))))
    rows = []

    for iteration in range(n_subsamples):
        sample_1 = rng.choice(terms, size=n_take, replace=False).tolist()
        sample_2 = rng.choice(terms, size=n_take, replace=False).tolist()
        common_terms = sorted(set(sample_1).intersection(sample_2))

        if len(common_terms) < max(3, k):
            rows.append({
                "k": k,
                "iteration": iteration,
                "n_common": len(common_terms),
                "ari": np.nan,
            })
            continue

        labels_1 = labels_from_submatrix(
            full_distance, sample_1, k, method=method
        ).loc[common_terms]
        labels_2 = labels_from_submatrix(
            full_distance, sample_2, k, method=method
        ).loc[common_terms]

        rows.append({
            "k": k,
            "iteration": iteration,
            "n_common": len(common_terms),
            "ari": adjusted_rand_score(labels_1, labels_2),
        })

    return pd.DataFrame(rows)


def validation_for_k(
    full_distance: pd.DataFrame,
    labels: pd.Series,
    k: int,
) -> dict:
    D = full_distance.to_numpy(dtype=float)
    labs = labels.loc[full_distance.index].to_numpy()
    n_observed_clusters = len(np.unique(labs))

    silhouette = (
        silhouette_score(D, labs, metric="precomputed")
        if 1 < n_observed_clusters < len(labs)
        else np.nan
    )
    sizes = labels.value_counts()

    return {
        "k": int(k),
        "n_observed_clusters": int(n_observed_clusters),
        "silhouette_mindist": float(silhouette) if pd.notna(silhouette) else np.nan,
        "min_cluster_size": int(sizes.min()),
        "max_cluster_size": int(sizes.max()),
        "median_cluster_size": float(sizes.median()),
    }


# Candidate-k validation sweep. This evaluates FINAL_K but does not replace it.
validation_rows = []
stability_tables = []

for k in K_RANGE:
    if not 2 <= k < len(mindist_tiebroken):
        continue

    labels_k = hierarchical_labels(
        mindist_tiebroken,
        k=k,
        method=LINKAGE_METHOD,
    )
    validation_rows.append(validation_for_k(mindist_tiebroken, labels_k, k))
    stability_tables.append(
        stability_for_k(
            mindist_tiebroken,
            k=k,
            n_subsamples=STABILITY_SUBSAMPLES,
            fraction=STABILITY_FRACTION,
            seed=RANDOM_STATE + k,
            method=LINKAGE_METHOD,
        )
    )

if not validation_rows:
    raise ValueError("K_RANGE contains no valid candidate values.")

validation_df = pd.DataFrame(validation_rows)
stability_raw = pd.concat(stability_tables, ignore_index=True)
stability_summary = (
    stability_raw
    .groupby("k", as_index=False)
    .agg(
        stability_mean=("ari", "mean"),
        stability_median=("ari", "median"),
        stability_p10=("ari", lambda x: x.quantile(0.10)),
        stability_p90=("ari", lambda x: x.quantile(0.90)),
        mean_common_terms=("n_common", "mean"),
        valid_stability_runs=("ari", "count"),
    )
)

cluster_selection = (
    validation_df
    .merge(stability_summary, on="k", how="left")
    .sort_values("k")
    .reset_index(drop=True)
)
cluster_selection["is_final_k"] = cluster_selection["k"].eq(FINAL_K)
cluster_selection["is_candidate_k"] = cluster_selection["k"].isin(CANDIDATE_K_VALUES)

cluster_selection.to_csv(
    VALIDATION_DIR / "cluster_selection_by_k.csv",
    index=False,
)
stability_raw.to_csv(
    VALIDATION_DIR / "subsample_stability_raw.csv",
    index=False,
)

print("Candidate cluster validation:")
print(
    cluster_selection
    .sort_values(
        ["stability_median", "silhouette_mindist"],
        ascending=[False, False],
        na_position="last",
    )
    .to_string(index=False)
)


def select_recommended_k(
    selection_df: pd.DataFrame,
    min_cluster_size: int,
) -> int:
    eligible = selection_df[
        selection_df["min_cluster_size"] >= min_cluster_size
    ]
    if eligible.empty:
        eligible = selection_df
    best = eligible.sort_values(
        ["stability_median", "silhouette_mindist"],
        ascending=[False, False],
        na_position="last",
    ).iloc[0]
    return int(best["k"])


RECOMMENDED_K = select_recommended_k(
    cluster_selection,
    MIN_ACCEPTABLE_CLUSTER_SIZE,
)

k_selection_audit = pd.DataFrame({
    "metric": [
        "configured_final_k",
        "recommended_k",
        "min_acceptable_cluster_size",
        "configured_k_matches_recommendation",
    ],
    "value": [
        FINAL_K,
        RECOMMENDED_K,
        MIN_ACCEPTABLE_CLUSTER_SIZE,
        FINAL_K == RECOMMENDED_K,
    ],
})

k_selection_audit.to_csv(
    VALIDATION_DIR / "k_selection_audit.csv",
    index=False,
)

print(
    f"\nConfigured FINAL_K={FINAL_K}; validation recommends "
    f"k={RECOMMENDED_K}. The configured baseline is not changed."
)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(
    cluster_selection["k"],
    cluster_selection["stability_median"],
    marker="o",
    label="Median subsample stability",
)
ax.plot(
    cluster_selection["k"],
    cluster_selection["silhouette_mindist"],
    marker="s",
    label="Silhouette score",
)
ax.axvline(
    FINAL_K,
    linestyle="--",
    linewidth=1.5,
    label=f"Final k = {FINAL_K}",
)
ax.set_title("Cluster validation across candidate values of k")
ax.set_xlabel("Number of clusters, k")
ax.set_ylabel("Validation score")
ax.set_xticks(cluster_selection["k"])
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(
    VALIDATION_DIR / "cluster_selection_scores.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


def term_stability_from_distance(
    full_distance: pd.DataFrame,
    final_labels: pd.Series,
) -> pd.DataFrame:
    rows = []
    aligned_labels = final_labels.loc[full_distance.index]

    for term in aligned_labels.index:
        cluster_id = aligned_labels.loc[term]
        same_terms = aligned_labels.index[
            (aligned_labels == cluster_id) & (aligned_labels.index != term)
        ]
        other_terms = aligned_labels.index[aligned_labels != cluster_id]

        intra_dist = (
            full_distance.loc[term, same_terms].mean()
            if len(same_terms) > 0
            else np.nan
        )
        between_dist = (
            full_distance.loc[term, other_terms].mean()
            if len(other_terms) > 0
            else np.nan
        )
        margin = (
            between_dist - intra_dist
            if pd.notna(intra_dist) and pd.notna(between_dist)
            else np.nan
        )

        rows.append({
            "term": term,
            "cluster": int(cluster_id),
            "mean_intra_mindist": intra_dist,
            "mean_between_mindist": between_dist,
            "mindist_margin": margin,
        })

    term_df = pd.DataFrame(rows)
    term_df["term_stability"] = (
        term_df
        .groupby("cluster")["mindist_margin"]
        .rank(pct=True, method="average")
    )
    return term_df.sort_values(
        ["cluster", "term_stability", "mindist_margin"],
        ascending=[True, False, False],
    ).reset_index(drop=True)


def select_core_terms(
    term_stability: pd.DataFrame,
    top_n: int = TOP_N_CORE_TERMS,
) -> pd.DataFrame:
    selected = []

    for cluster_id, group in term_stability.groupby("cluster"):
        group = group.sort_values(
            ["term_stability", "mindist_margin"],
            ascending=[False, False],
        ).copy()
        cutoff = group["term_stability"].quantile(CORE_STABILITY_QUANTILE)
        core = group[group["term_stability"] >= cutoff].copy()

        min_keep = min(MIN_CORE_TERMS, len(group))
        if len(core) < min_keep:
            core = group.head(min_keep).copy()
        if top_n is not None and top_n > 0:
            core = core.head(max(top_n, min_keep)).copy()

        core["core_rank"] = np.arange(1, len(core) + 1)
        selected.append(core)

    return pd.concat(selected, ignore_index=True)


def weighted_average(panel: pd.DataFrame, weights: pd.Series) -> pd.Series:
    columns = [column for column in weights.index if column in panel.columns]
    if not columns:
        return pd.Series(np.nan, index=panel.index, dtype=float)

    aligned_weights = weights.loc[columns].astype(float).fillna(0.0)
    if aligned_weights.sum() <= 0:
        aligned_weights = pd.Series(1.0 / len(columns), index=columns)
    else:
        aligned_weights = aligned_weights / aligned_weights.sum()

    return panel[columns].mul(aligned_weights, axis=1).sum(axis=1)


def interpret_solution(
    k: int,
    labels: pd.Series,
    save_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build term-level stability, core terms, and a cluster summary for one
    already-fitted candidate k (from Step 4's CLUSTERING_BY_K), without
    recomputing its cluster assignments.
    """
    save_dir.mkdir(parents=True, exist_ok=True)

    term_stability_k = term_stability_from_distance(mindist, labels)
    core_terms_k = select_core_terms(term_stability_k, top_n=TOP_N_CORE_TERMS)

    term_stability_k.to_csv(save_dir / "term_stability.csv", index=False)
    core_terms_k.to_csv(save_dir / "core_terms.csv", index=False)

    summary_k = (
        term_stability_k
        .groupby("cluster", as_index=False)
        .agg(
            n_terms=("term", "size"),
            median_term_stability=("term_stability", "median"),
            mean_intra_mindist=("mean_intra_mindist", "mean"),
            mean_between_mindist=("mean_between_mindist", "mean"),
            mean_mindist_margin=("mindist_margin", "mean"),
        )
    )
    top_terms_k = (
        core_terms_k
        .sort_values(["cluster", "core_rank"])
        .groupby("cluster")["term"]
        .apply(lambda terms: ", ".join(terms.head(TOP_N_CORE_TERMS)))
    )
    summary_k["top_10_core_terms"] = summary_k["cluster"].map(top_terms_k)

    expected_sizes = labels.value_counts().sort_index()
    summary_k["expected_n_terms"] = summary_k["cluster"].map(expected_sizes)
    summary_k["size_matches_labels"] = (
        summary_k["n_terms"] == summary_k["expected_n_terms"]
    )
    if not summary_k["size_matches_labels"].all():
        raise RuntimeError(f"Cluster-summary sizes do not match labels for k={k}.")

    summary_k.to_csv(save_dir / "cluster_summary.csv", index=False)
    return term_stability_k, core_terms_k, summary_k


# Interpret every candidate fit in Step 4 -- reuses CLUSTERING_BY_K, so no
# cluster assignment is recomputed here, only its downstream interpretation.
INTERPRETATION_BY_K = {
    k: interpret_solution(
        k, CLUSTERING_BY_K[k]["labels"], VALIDATION_DIR / f"k_{k}"
    )
    for k in CANDIDATE_K_VALUES
}

candidate_k_interpretation = pd.DataFrame([
    {
        "k": k,
        "n_clusters": len(summary_k),
        "weakest_cluster_margin": float(summary_k["mean_mindist_margin"].min()),
        "strongest_cluster_margin": float(summary_k["mean_mindist_margin"].max()),
        "largest_cluster_n_terms": int(summary_k["n_terms"].max()),
        "is_final_k": k == FINAL_K,
    }
    for k, (_, _, summary_k) in INTERPRETATION_BY_K.items()
])
candidate_k_interpretation.to_csv(
    VALIDATION_DIR / "candidate_k_interpretation_summary.csv", index=False
)

print("Interpretive comparison across candidate k values:")
print(
    "(weakest_cluster_margin close to 0 or negative flags a cluster that is "
    "not well separated -- see cluster_summary.csv in that k's folder)\n"
)
print(candidate_k_interpretation.to_string(index=False))

# Promote FINAL_K's interpretation as the notebook-wide primary summary, and
# mirror it under flat, k-independent filenames for discoverability -- its
# authoritative copy still lives in k_{FINAL_K}/.
term_stability, core_terms, cluster_summary = INTERPRETATION_BY_K[FINAL_K]
term_stability.to_csv(VALIDATION_DIR / "term_stability.csv", index=False)
core_terms.to_csv(VALIDATION_DIR / "core_terms.csv", index=False)
cluster_summary.to_csv(VALIDATION_DIR / "cluster_summary.csv", index=False)

print(f"\nPrimary cluster interpretation (FINAL_K={FINAL_K}):")
print(cluster_summary.to_string(index=False))

Candidate cluster validation:
 k  n_observed_clusters  silhouette_mindist  min_cluster_size  max_cluster_size  median_cluster_size  stability_mean  stability_median  stability_p10  stability_p90  mean_common_terms  valid_stability_runs  is_final_k  is_candidate_k
 3                    3            0.060385                58               613                176.0        0.382001          0.387509       0.152782       0.614605           542.3375                    80       False           False
 8                    8            0.035004                25               436                 65.0        0.395318          0.379658       0.294352       0.503064           543.6125                    80       False            True
 9                    9            0.042690                25               301                 72.0        0.360076          0.357238       0.294452       0.434958           544.0375                    80       False            True
12                   12           

## 6. Robustness and sensitivity analysis

This section evaluates whether the configured baseline partition
(`labels_final`, `FINAL_K`, and `LINKAGE_METHOD`) is defensible using
three complementary diagnostics.

1. **Consensus matrix** — repeatedly subsample the terms, recluster
   them, and record how often each pair is assigned to the same cluster.
   Stable clusters appear as crisp, high-consensus diagonal blocks,
   whereas unstable boundaries appear as diffuse intermediate values.

2. **Multi-\(k\) stability** — reuse the subsample ARI sweep already
   calculated in Step 5 through `cluster_selection` and
   `stability_raw`. This compares partition stability across candidate
   values of \(k\) without recomputing the validation sweep.

3. **Silhouette diagnostics** — calculate global, cluster-level, and
   term-level silhouette values from the precomputed MINDIST matrix.
   The baseline linkage method is compared with every method in
   `LINKAGE_CANDIDATES` at the same configured value of `FINAL_K`.

These diagnostics measure different properties. Consensus and
subsample ARI measure reproducibility under resampling, while
silhouette measures within-cluster cohesion and between-cluster
separation. None of the checks automatically replaces the configured
baseline partition.

The section reuses the existing distance matrix, labels, candidate-\(k\)
results, linkage configuration, random seed, and output paths defined
in Steps 0–5.

In [35]:
# ---------------------------------------------------------------------
# STEP 6: ROBUSTNESS AND SENSITIVITY ANALYSIS
# ---------------------------------------------------------------------

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_samples, silhouette_score

ROBUSTNESS_DIR = OUTPUT_DIR / "06_robustness"
ROBUSTNESS_DIR.mkdir(parents=True, exist_ok=True)

# Use the existing Step 5 stability settings unless a separate
# consensus configuration has already been declared.
CONSENSUS_SUBSAMPLES = int(
    globals().get("CONSENSUS_SUBSAMPLES", STABILITY_SUBSAMPLES)
)
CONSENSUS_FRACTION = float(
    globals().get("CONSENSUS_FRACTION", STABILITY_FRACTION)
)

if CONSENSUS_SUBSAMPLES < 2:
    raise ValueError("CONSENSUS_SUBSAMPLES must be at least 2.")

if not 0 < CONSENSUS_FRACTION < 1:
    raise ValueError(
        "CONSENSUS_FRACTION must be strictly between 0 and 1."
    )


# ---------------------------------------------------------------------
# Validate and align the baseline objects produced in Steps 3–5
# ---------------------------------------------------------------------

if not isinstance(mindist_tiebroken, pd.DataFrame):
    raise TypeError(
        "mindist_tiebroken must be a pandas DataFrame."
    )

if mindist_tiebroken.shape[0] != mindist_tiebroken.shape[1]:
    raise ValueError("mindist_tiebroken must be square.")

if not mindist_tiebroken.index.equals(
    mindist_tiebroken.columns
):
    raise ValueError(
        "mindist_tiebroken rows and columns must have the "
        "same terms in the same order."
    )

distance_matrix = mindist_tiebroken.to_numpy(dtype=float)

if not np.isfinite(distance_matrix).all():
    raise ValueError(
        "mindist_tiebroken contains NaN or infinite values."
    )

if not np.allclose(
    distance_matrix,
    distance_matrix.T,
    atol=1e-10,
    rtol=0,
):
    raise ValueError("mindist_tiebroken must be symmetric.")

if not np.allclose(
    np.diag(distance_matrix),
    0.0,
    atol=1e-10,
    rtol=0,
):
    raise ValueError(
        "mindist_tiebroken must have a zero diagonal."
    )

if not isinstance(labels_final, pd.Series):
    labels_final = pd.Series(
        labels_final,
        index=mindist_tiebroken.index,
        name="cluster",
    )

missing_terms = mindist_tiebroken.index.difference(
    labels_final.index
)
extra_terms = labels_final.index.difference(
    mindist_tiebroken.index
)

if len(missing_terms) > 0 or len(extra_terms) > 0:
    raise ValueError(
        "labels_final and mindist_tiebroken must contain exactly "
        "the same terms."
    )

labels_final = (
    labels_final
    .reindex(mindist_tiebroken.index)
    .astype(int)
)

if labels_final.nunique() != FINAL_K:
    raise ValueError(
        f"FINAL_K={FINAL_K}, but labels_final contains "
        f"{labels_final.nunique()} clusters."
    )


# ---------------------------------------------------------------------
# Shared helper: cluster a precomputed distance matrix
# ---------------------------------------------------------------------

def labels_from_precomputed_distance(
    distance_df: pd.DataFrame,
    k: int,
    method: str,
) -> pd.Series:
    """
    Cluster a square precomputed distance matrix.

    The implementation follows the notebook's existing convention:
    scipy linkage is applied to the condensed distance vector for all
    candidate linkage methods, including the configured baseline.
    """
    if not 2 <= k < len(distance_df):
        raise ValueError(
            f"k must be between 2 and {len(distance_df) - 1}; "
            f"received {k}."
        )

    condensed = squareform(
        distance_df.to_numpy(dtype=float),
        checks=False,
    )
    linkage_matrix = linkage(condensed, method=method)

    return pd.Series(
        fcluster(
            linkage_matrix,
            t=k,
            criterion="maxclust",
        ),
        index=distance_df.index,
        name="cluster",
    ).astype(int)


# =====================================================================
# 6.1 Consensus matrix
# =====================================================================

def calculate_consensus_matrix(
    full_distance: pd.DataFrame,
    k: int,
    method: str,
    n_subsamples: int,
    sample_fraction: float,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Estimate pairwise clustering consensus under term subsampling.

    For each iteration, a subset of terms is drawn without replacement
    and reclustered using the configured k and linkage method.

    Returns
    -------
    consensus_df
        Pairwise co-clustering count divided by pairwise co-occurrence
        count.

    cocluster_count_df
        Number of iterations in which each pair was placed in the same
        cluster.

    cooccurrence_count_df
        Number of iterations in which each pair was sampled together.
    """
    terms = full_distance.index
    n_terms = len(terms)

    n_take = int(round(n_terms * sample_fraction))
    n_take = max(k + 2, n_take)
    n_take = min(n_take, n_terms - 1)

    if n_take <= k:
        raise ValueError(
            "The consensus subsample size must be greater than k."
        )

    rng = np.random.default_rng(seed)

    cooccurrence = np.zeros(
        (n_terms, n_terms),
        dtype=np.int32,
    )
    coclustering = np.zeros(
        (n_terms, n_terms),
        dtype=np.int32,
    )

    position_lookup = pd.Series(
        np.arange(n_terms),
        index=terms,
    )

    for iteration in range(n_subsamples):
        sampled_terms = pd.Index(
            rng.choice(
                terms.to_numpy(),
                size=n_take,
                replace=False,
            )
        )

        sampled_positions = (
            position_lookup
            .loc[sampled_terms]
            .to_numpy(dtype=int)
        )

        # Record that all pairs in the current subset co-occurred.
        cooccurrence[
            np.ix_(sampled_positions, sampled_positions)
        ] += 1

        subsample_distance = full_distance.loc[
            sampled_terms,
            sampled_terms,
        ]

        subsample_labels = labels_from_precomputed_distance(
            distance_df=subsample_distance,
            k=k,
            method=method,
        )

        # Record co-clustering separately within each recovered cluster.
        for cluster_id in subsample_labels.unique():
            cluster_terms = subsample_labels.index[
                subsample_labels == cluster_id
            ]
            cluster_positions = (
                position_lookup
                .loc[cluster_terms]
                .to_numpy(dtype=int)
            )

            coclustering[
                np.ix_(cluster_positions, cluster_positions)
            ] += 1

    consensus = np.divide(
        coclustering,
        cooccurrence,
        out=np.full(
            coclustering.shape,
            np.nan,
            dtype=float,
        ),
        where=cooccurrence > 0,
    )

    # A term is always perfectly consistent with itself.
    np.fill_diagonal(consensus, 1.0)

    consensus_df = pd.DataFrame(
        consensus,
        index=terms,
        columns=terms,
    )
    cocluster_count_df = pd.DataFrame(
        coclustering,
        index=terms,
        columns=terms,
    )
    cooccurrence_count_df = pd.DataFrame(
        cooccurrence,
        index=terms,
        columns=terms,
    )

    return (
        consensus_df,
        cocluster_count_df,
        cooccurrence_count_df,
    )


(
    consensus_matrix,
    consensus_cocluster_counts,
    consensus_cooccurrence_counts,
) = calculate_consensus_matrix(
    full_distance=mindist_tiebroken,
    k=FINAL_K,
    method=LINKAGE_METHOD,
    n_subsamples=CONSENSUS_SUBSAMPLES,
    sample_fraction=CONSENSUS_FRACTION,
    seed=RANDOM_STATE,
)

consensus_matrix.to_csv(
    ROBUSTNESS_DIR / "consensus_matrix.csv"
)
consensus_cocluster_counts.to_csv(
    ROBUSTNESS_DIR / "consensus_cocluster_counts.csv"
)
consensus_cooccurrence_counts.to_csv(
    ROBUSTNESS_DIR / "consensus_cooccurrence_counts.csv"
)


# Order terms by the baseline dendrogram rather than alphabetically.
if (
    "Z_SHARED" in globals()
    and len(Z_SHARED) == len(mindist_tiebroken) - 1
):
    baseline_linkage = Z_SHARED
else:
    baseline_linkage = linkage(
        squareform(distance_matrix, checks=False),
        method=LINKAGE_METHOD,
    )

baseline_leaf_order = dendrogram(
    baseline_linkage,
    no_plot=True,
)["leaves"]

ordered_terms = mindist_tiebroken.index[
    baseline_leaf_order
]
consensus_ordered = consensus_matrix.loc[
    ordered_terms,
    ordered_terms,
]

fig, ax = plt.subplots(figsize=(10, 9))
image = ax.imshow(
    consensus_ordered.to_numpy(dtype=float),
    vmin=0,
    vmax=1,
    aspect="auto",
    interpolation="nearest",
)
ax.set_title(
    "Subsample consensus matrix\n"
    f"k={FINAL_K}, linkage={LINKAGE_METHOD}, "
    f"subsamples={CONSENSUS_SUBSAMPLES}"
)
ax.set_xlabel("Terms ordered by baseline dendrogram")
ax.set_ylabel("Terms ordered by baseline dendrogram")
ax.set_xticks([])
ax.set_yticks([])

colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
colorbar.set_label("Co-clustering probability")

fig.tight_layout()
fig.savefig(
    ROBUSTNESS_DIR / "consensus_matrix.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


def summarize_consensus_by_cluster(
    consensus_df: pd.DataFrame,
    baseline_labels: pd.Series,
) -> pd.DataFrame:
    """
    Summarize within-cluster and nearest between-cluster consensus.
    """
    rows = []
    cluster_ids = sorted(baseline_labels.unique())

    for cluster_id in cluster_ids:
        own_terms = baseline_labels.index[
            baseline_labels == cluster_id
        ]
        other_terms = baseline_labels.index[
            baseline_labels != cluster_id
        ]

        own_values = consensus_df.loc[
            own_terms,
            own_terms,
        ].to_numpy(dtype=float)

        # Remove the diagonal, which is fixed at one.
        own_values = own_values[
            ~np.eye(
                len(own_terms),
                dtype=bool,
            )
        ]
        own_values = own_values[np.isfinite(own_values)]

        between_by_cluster = {}

        for alternative_id in cluster_ids:
            if alternative_id == cluster_id:
                continue

            alternative_terms = baseline_labels.index[
                baseline_labels == alternative_id
            ]

            values = consensus_df.loc[
                own_terms,
                alternative_terms,
            ].to_numpy(dtype=float)
            values = values[np.isfinite(values)]

            between_by_cluster[alternative_id] = (
                float(np.mean(values))
                if len(values) > 0
                else np.nan
            )

        valid_between = {
            key: value
            for key, value in between_by_cluster.items()
            if pd.notna(value)
        }

        nearest_cluster = (
            max(valid_between, key=valid_between.get)
            if valid_between
            else np.nan
        )
        maximum_between = (
            valid_between[nearest_cluster]
            if valid_between
            else np.nan
        )

        mean_within = (
            float(np.mean(own_values))
            if len(own_values) > 0
            else np.nan
        )

        rows.append(
            {
                "cluster": int(cluster_id),
                "n_terms": int(len(own_terms)),
                "mean_within_consensus": mean_within,
                "median_within_consensus": (
                    float(np.median(own_values))
                    if len(own_values) > 0
                    else np.nan
                ),
                "p10_within_consensus": (
                    float(np.quantile(own_values, 0.10))
                    if len(own_values) > 0
                    else np.nan
                ),
                "nearest_competing_cluster": nearest_cluster,
                "max_mean_between_consensus": maximum_between,
                "consensus_margin": (
                    mean_within - maximum_between
                    if pd.notna(mean_within)
                    and pd.notna(maximum_between)
                    else np.nan
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            "mean_within_consensus",
            ascending=True,
        )
        .reset_index(drop=True)
    )


consensus_cluster_summary = summarize_consensus_by_cluster(
    consensus_df=consensus_matrix,
    baseline_labels=labels_final,
)
consensus_cluster_summary.to_csv(
    ROBUSTNESS_DIR / "consensus_cluster_summary.csv",
    index=False,
)


# =====================================================================
# 6.2 Multi-k stability: reuse Step 5 outputs
# =====================================================================

required_selection_columns = {
    "k",
    "stability_mean",
    "stability_median",
    "stability_p10",
    "stability_p90",
    "silhouette_mindist",
}

missing_selection_columns = (
    required_selection_columns - set(cluster_selection.columns)
)

if missing_selection_columns:
    raise KeyError(
        "cluster_selection is missing required Step 5 columns: "
        f"{sorted(missing_selection_columns)}"
    )

multi_k_stability = (
    cluster_selection
    .copy()
    .sort_values("k")
    .reset_index(drop=True)
)

multi_k_stability["is_final_k"] = (
    multi_k_stability["k"] == FINAL_K
)
multi_k_stability.to_csv(
    ROBUSTNESS_DIR / "multi_k_stability_summary.csv",
    index=False,
)

# Preserve the already-computed iteration-level results under Step 6.
if "stability_raw" in globals():
    stability_raw.to_csv(
        ROBUSTNESS_DIR / "multi_k_stability_raw.csv",
        index=False,
    )

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    multi_k_stability["k"],
    multi_k_stability["stability_median"],
    marker="o",
    label="Median subsample ARI",
)

ax.fill_between(
    multi_k_stability["k"].to_numpy(dtype=float),
    multi_k_stability["stability_p10"].to_numpy(dtype=float),
    multi_k_stability["stability_p90"].to_numpy(dtype=float),
    alpha=0.20,
    label="10th–90th percentile",
)

ax.axvline(
    FINAL_K,
    linestyle="--",
    linewidth=1.5,
    label=f"Configured k = {FINAL_K}",
)

ax.set_title("Subsample stability across candidate values of k")
ax.set_xlabel("Number of clusters, k")
ax.set_ylabel("Adjusted Rand index")
ax.set_xticks(multi_k_stability["k"])
ax.grid(alpha=0.25)
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(
    ROBUSTNESS_DIR / "multi_k_stability.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


# =====================================================================
# 6.3 Silhouette diagnostics at FINAL_K
# =====================================================================

def silhouette_diagnostics(
    distance_df: pd.DataFrame,
    labels: pd.Series,
    method: str,
) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    """
    Calculate global, cluster-level, and term-level silhouette results.
    """
    aligned_labels = (
        labels
        .reindex(distance_df.index)
        .astype(int)
    )
    label_values = aligned_labels.to_numpy()
    n_clusters = aligned_labels.nunique()

    if not 1 < n_clusters < len(aligned_labels):
        raise ValueError(
            "Silhouette requires at least two clusters and fewer "
            "clusters than observations."
        )

    term_values = silhouette_samples(
        distance_df.to_numpy(dtype=float),
        label_values,
        metric="precomputed",
    )

    term_df = pd.DataFrame(
        {
            "term": distance_df.index,
            "cluster": label_values,
            "silhouette": term_values,
            "linkage_method": method,
        }
    )

    cluster_df = (
        term_df
        .groupby("cluster", as_index=False)
        .agg(
            n_terms=("term", "size"),
            silhouette_mean=("silhouette", "mean"),
            silhouette_median=("silhouette", "median"),
            silhouette_p10=(
                "silhouette",
                lambda values: values.quantile(0.10),
            ),
            silhouette_p90=(
                "silhouette",
                lambda values: values.quantile(0.90),
            ),
            negative_silhouette_share=(
                "silhouette",
                lambda values: float((values < 0).mean()),
            ),
            near_zero_silhouette_share=(
                "silhouette",
                lambda values: float(
                    ((values >= 0) & (values < 0.10)).mean()
                ),
            ),
        )
        .sort_values("silhouette_mean")
        .reset_index(drop=True)
    )
    cluster_df["linkage_method"] = method

    global_result = {
        "linkage_method": method,
        "k": int(n_clusters),
        "global_silhouette": float(
            silhouette_score(
                distance_df.to_numpy(dtype=float),
                label_values,
                metric="precomputed",
            )
        ),
        "minimum_cluster_silhouette": float(
            cluster_df["silhouette_mean"].min()
        ),
        "median_cluster_silhouette": float(
            cluster_df["silhouette_mean"].median()
        ),
        "negative_term_share": float(
            (term_df["silhouette"] < 0).mean()
        ),
        "minimum_cluster_size": int(
            cluster_df["n_terms"].min()
        ),
        "maximum_cluster_size": int(
            cluster_df["n_terms"].max()
        ),
    }

    return global_result, cluster_df, term_df


linkage_methods = list(
    dict.fromkeys(
        [LINKAGE_METHOD] + list(LINKAGE_CANDIDATES)
    )
)

linkage_global_rows = []
linkage_cluster_tables = []
linkage_term_tables = []
linkage_partitions = {}

for method in linkage_methods:
    if method == LINKAGE_METHOD:
        method_labels = labels_final.copy()
    else:
        method_labels = labels_from_precomputed_distance(
            distance_df=mindist_tiebroken,
            k=FINAL_K,
            method=method,
        )

    linkage_partitions[method] = method_labels

    (
        global_result,
        cluster_result,
        term_result,
    ) = silhouette_diagnostics(
        distance_df=mindist_tiebroken,
        labels=method_labels,
        method=method,
    )

    global_result["is_baseline_linkage"] = (
        method == LINKAGE_METHOD
    )
    global_result["ari_vs_baseline"] = (
        1.0
        if method == LINKAGE_METHOD
        else float(
            adjusted_rand_score(
                labels_final,
                method_labels.reindex(labels_final.index),
            )
        )
    )

    linkage_global_rows.append(global_result)
    linkage_cluster_tables.append(cluster_result)
    linkage_term_tables.append(term_result)

linkage_silhouette_summary = (
    pd.DataFrame(linkage_global_rows)
    .sort_values(
        "global_silhouette",
        ascending=False,
    )
    .reset_index(drop=True)
)

linkage_cluster_silhouette = pd.concat(
    linkage_cluster_tables,
    ignore_index=True,
)

linkage_term_silhouette = pd.concat(
    linkage_term_tables,
    ignore_index=True,
)

linkage_silhouette_summary.to_csv(
    ROBUSTNESS_DIR / "linkage_silhouette_summary.csv",
    index=False,
)
linkage_cluster_silhouette.to_csv(
    ROBUSTNESS_DIR / "linkage_cluster_silhouette.csv",
    index=False,
)
linkage_term_silhouette.to_csv(
    ROBUSTNESS_DIR / "linkage_term_silhouette.csv",
    index=False,
)

linkage_assignments = pd.concat(
    [
        labels.rename(f"cluster_{method}")
        for method, labels in linkage_partitions.items()
    ],
    axis=1,
)
linkage_assignments.index.name = "term"
linkage_assignments.to_csv(
    ROBUSTNESS_DIR / "linkage_cluster_assignments.csv"
)


fig, ax = plt.subplots(figsize=(8.5, 5.2))

plot_summary = linkage_silhouette_summary.sort_values(
    "global_silhouette",
    ascending=True,
)

ax.barh(
    plot_summary["linkage_method"],
    plot_summary["global_silhouette"],
)
ax.axvline(0, linewidth=1)
ax.set_title(
    f"Silhouette comparison at fixed k = {FINAL_K}"
)
ax.set_xlabel("Global silhouette score")
ax.set_ylabel("Linkage method")

fig.tight_layout()
fig.savefig(
    ROBUSTNESS_DIR / "linkage_silhouette_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


# Baseline cluster-level silhouette plot.
baseline_cluster_silhouette = (
    linkage_cluster_silhouette[
        linkage_cluster_silhouette["linkage_method"]
        == LINKAGE_METHOD
    ]
    .sort_values("silhouette_mean")
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.barh(
    baseline_cluster_silhouette["cluster"].astype(str),
    baseline_cluster_silhouette["silhouette_mean"],
)
ax.axvline(0, linewidth=1)
ax.set_title(
    "Baseline cluster-level silhouette scores\n"
    f"k={FINAL_K}, linkage={LINKAGE_METHOD}"
)
ax.set_xlabel("Mean silhouette score")
ax.set_ylabel("Cluster")

fig.tight_layout()
fig.savefig(
    ROBUSTNESS_DIR / "baseline_cluster_silhouette.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


# =====================================================================
# Consolidated audit table
# =====================================================================

baseline_linkage_row = linkage_silhouette_summary.loc[
    linkage_silhouette_summary["linkage_method"]
    == LINKAGE_METHOD
].iloc[0]

final_k_row = multi_k_stability.loc[
    multi_k_stability["k"] == FINAL_K
]

if final_k_row.empty:
    final_k_stability_median = np.nan
    final_k_stability_p10 = np.nan
    final_k_stability_p90 = np.nan
else:
    final_k_row = final_k_row.iloc[0]
    final_k_stability_median = final_k_row[
        "stability_median"
    ]
    final_k_stability_p10 = final_k_row[
        "stability_p10"
    ]
    final_k_stability_p90 = final_k_row[
        "stability_p90"
    ]

robustness_audit = pd.DataFrame(
    [
        {
            "diagnostic": "consensus",
            "metric": "mean cluster-level within consensus",
            "value": consensus_cluster_summary[
                "mean_within_consensus"
            ].mean(),
        },
        {
            "diagnostic": "consensus",
            "metric": "weakest cluster within consensus",
            "value": consensus_cluster_summary[
                "mean_within_consensus"
            ].min(),
        },
        {
            "diagnostic": "multi_k_stability",
            "metric": "FINAL_K median subsample ARI",
            "value": final_k_stability_median,
        },
        {
            "diagnostic": "multi_k_stability",
            "metric": "FINAL_K 10th percentile ARI",
            "value": final_k_stability_p10,
        },
        {
            "diagnostic": "multi_k_stability",
            "metric": "FINAL_K 90th percentile ARI",
            "value": final_k_stability_p90,
        },
        {
            "diagnostic": "silhouette",
            "metric": "baseline global silhouette",
            "value": baseline_linkage_row[
                "global_silhouette"
            ],
        },
        {
            "diagnostic": "silhouette",
            "metric": "weakest baseline cluster silhouette",
            "value": baseline_linkage_row[
                "minimum_cluster_silhouette"
            ],
        },
        {
            "diagnostic": "silhouette",
            "metric": "baseline negative term share",
            "value": baseline_linkage_row[
                "negative_term_share"
            ],
        },
    ]
)

robustness_audit.to_csv(
    ROBUSTNESS_DIR / "robustness_audit.csv",
    index=False,
)


# =====================================================================
# Console summary
# =====================================================================

print("=" * 78)
print("STEP 6: ROBUSTNESS AND SENSITIVITY ANALYSIS")
print("=" * 78)

print(
    "\n1. Weakest clusters by within-cluster consensus:"
)
print(
    consensus_cluster_summary[
        [
            "cluster",
            "n_terms",
            "mean_within_consensus",
            "p10_within_consensus",
            "max_mean_between_consensus",
            "consensus_margin",
        ]
    ]
    .head(FINAL_K)
    .to_string(
        index=False,
        float_format=lambda value: f"{value:.3f}",
    )
)

print("\n2. Multi-k stability reused from Step 5:")
print(
    multi_k_stability[
        [
            "k",
            "stability_mean",
            "stability_median",
            "stability_p10",
            "stability_p90",
            "silhouette_mindist",
            "is_final_k",
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda value: f"{value:.3f}",
    )
)

print(
    "\n3. Linkage-method silhouette comparison at "
    f"FINAL_K={FINAL_K}:"
)
print(
    linkage_silhouette_summary[
        [
            "linkage_method",
            "global_silhouette",
            "minimum_cluster_silhouette",
            "negative_term_share",
            "ari_vs_baseline",
            "is_baseline_linkage",
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda value: f"{value:.3f}",
    )
)

print(
    "\nBaseline cluster-level silhouette summary "
    "(weakest clusters first):"
)
print(
    baseline_cluster_silhouette[
        [
            "cluster",
            "n_terms",
            "silhouette_mean",
            "silhouette_median",
            "silhouette_p10",
            "negative_silhouette_share",
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda value: f"{value:.3f}",
    )
)

print(
    f"\nStep 6 outputs saved to: {ROBUSTNESS_DIR.resolve()}"
)

STEP 6: ROBUSTNESS AND SENSITIVITY ANALYSIS

1. Weakest clusters by within-cluster consensus:
 cluster  n_terms  mean_within_consensus  p10_within_consensus  max_mean_between_consensus  consensus_margin
       8      301                  0.374                 0.130                       0.312             0.062
      10       88                  0.403                 0.120                       0.328             0.075
       1       90                  0.580                 0.043                       0.142             0.438
       5       72                  0.672                 0.263                       0.085             0.587
       9       47                  0.707                 0.267                       0.328             0.379
       2       46                  0.750                 0.220                       0.080             0.669
       4       58                  0.762                 0.491                       0.055             0.706
       3       40                 

## 7. Visualizations

Produce one clear, standalone visualization for every cluster in the final clustering solution.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import periodogram, savgol_filter
from statsmodels.tsa.seasonal import STL


# =============================================================================
# 0. CONFIGURATION
# =============================================================================

@dataclass(frozen=True)
class ClusterShapeConfig:
    # Member representation
    max_terms_for_centroid: int = 25
    n_representative_terms: int = 3

    # Display smoothing
    min_smooth_days: int = 7
    max_smooth_days: int = 31

    # Local shape window
    zoom_window_days: int = 180
    zoom_step_days: int = 14

    # Periodicity and STL
    min_period_days: int = 7
    max_period_days: int = 550
    min_spectral_concentration: float = 4.0
    min_cycles_for_stl: float = 2.5

    # Figures
    figure_width: float = 14.0
    figure_height: float = 5.2
    stl_figure_width: float = 12.0
    stl_figure_height: float = 8.5
    dpi: int = 300


CFG = ClusterShapeConfig()

ROBUSTNESS_DIR = OUTPUT_DIR / "06_robustness"

CONSENSUS_PATH = (
    ROBUSTNESS_DIR
    / "consensus_cluster_summary.csv"
)

SILHOUETTE_PATH = (
    ROBUSTNESS_DIR
    / "linkage_cluster_silhouette.csv"
)

VIS_DIR = (
    OUTPUT_DIR
    / "07_visualization"
    / "all_cluster_shapes"
)

CLUSTER_FIG_DIR = VIS_DIR / "cluster_figures"
STL_DIR = VIS_DIR / "stl_decomposition"

VIS_DIR.mkdir(parents=True, exist_ok=True)
CLUSTER_FIG_DIR.mkdir(parents=True, exist_ok=True)
STL_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1. INPUT VALIDATION
# =============================================================================

def validate_panel(
    panel: pd.DataFrame,
) -> pd.DataFrame:
    """
    Validate and standardize the normalized time-series panel.

    Expected structure
    ------------------
    index:
        Dates or values convertible to dates.

    columns:
        Search-term identifiers.

    values:
        Numeric normalized search-interest observations.
    """
    if not isinstance(panel, pd.DataFrame):
        raise TypeError(
            "panel_norm must be a pandas DataFrame."
        )

    out = panel.copy()

    if not isinstance(out.index, pd.DatetimeIndex):
        out.index = pd.to_datetime(
            out.index,
            errors="coerce",
        )

    out = out.loc[~out.index.isna()]
    out = out.loc[
        ~out.index.duplicated(keep="first")
    ]
    out = out.sort_index()

    out.columns = out.columns.astype(str)
    out = out.apply(
        pd.to_numeric,
        errors="coerce",
    )

    if out.empty:
        raise ValueError(
            "The normalized panel is empty after validation."
        )

    if out.shape[1] == 0:
        raise ValueError(
            "The normalized panel contains no term columns."
        )

    return out


def normalize_labels_input(
    labels: pd.Series | pd.DataFrame,
) -> pd.DataFrame:
    """
    Standardize final cluster labels and optional term stability.

    Accepted Series
    ---------------
    index:
        term

    values:
        cluster

    Accepted DataFrame
    ------------------
    Required:
        cluster or equivalent cluster-label column

    Optional:
        term
        stability_score
    """
    if isinstance(labels, pd.Series):
        out = labels.rename("cluster").to_frame()
        out.index = out.index.astype(str)
        out.index.name = "term"

        out["cluster"] = pd.to_numeric(
            out["cluster"],
            errors="raise",
        ).astype(int)

        return out

    if not isinstance(labels, pd.DataFrame):
        raise TypeError(
            "labels_final must be a pandas Series "
            "or DataFrame."
        )

    out = labels.copy()

    if "term" in out.columns:
        out["term"] = out["term"].astype(str)
        out = out.set_index("term")
    else:
        out.index = out.index.astype(str)
        out.index.name = "term"

    cluster_candidates = [
        "cluster",
        "cluster_label",
        "consensus_cluster",
        "label",
    ]

    cluster_col = next(
        (
            col
            for col in cluster_candidates
            if col in out.columns
        ),
        None,
    )

    if cluster_col is None:
        raise ValueError(
            "Could not identify a cluster column. "
            f"Expected one of {cluster_candidates}."
        )

    if cluster_col != "cluster":
        out = out.rename(
            columns={cluster_col: "cluster"}
        )

    stability_candidates = [
        "stability_score",
        "term_stability",
        "consensus_score",
        "within_consensus",
    ]

    stability_col = next(
        (
            col
            for col in stability_candidates
            if col in out.columns
        ),
        None,
    )

    keep_columns = ["cluster"]

    if stability_col is not None:
        if stability_col != "stability_score":
            out = out.rename(
                columns={
                    stability_col: "stability_score"
                }
            )

        keep_columns.append("stability_score")

    out = out[keep_columns].copy()

    out["cluster"] = pd.to_numeric(
        out["cluster"],
        errors="raise",
    ).astype(int)

    if "stability_score" in out.columns:
        out["stability_score"] = pd.to_numeric(
            out["stability_score"],
            errors="coerce",
        )

    if out.index.duplicated().any():
        duplicated_terms = (
            out.index[
                out.index.duplicated(keep=False)
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            "Duplicate terms were found in labels_final. "
            f"Examples: {duplicated_terms[:10]}"
        )

    return out


panel_clean = validate_panel(panel_norm)
labels_table = normalize_labels_input(labels_final)


# =============================================================================
# 2. LOAD EXACT ROBUSTNESS TABLES
# =============================================================================

consensus_columns = [
    "cluster",
    "n_terms",
    "mean_within_consensus",
    "p10_within_consensus",
    "max_mean_between_consensus",
    "consensus_margin",
]

silhouette_columns = [
    "cluster",
    "silhouette_mean",
    "silhouette_median",
    "silhouette_p10",
    "negative_silhouette_share",
]


def require_columns(
    df: pd.DataFrame,
    required_columns: list[str],
    table_name: str,
) -> None:
    """Raise a clear error when a table lacks required fields."""
    missing = set(required_columns).difference(
        df.columns
    )

    if missing:
        raise ValueError(
            f"{table_name} is missing columns: "
            f"{sorted(missing)}"
        )


if not CONSENSUS_PATH.exists():
    raise FileNotFoundError(
        f"Consensus file not found:\n{CONSENSUS_PATH}"
    )

if not SILHOUETTE_PATH.exists():
    raise FileNotFoundError(
        f"Silhouette file not found:\n{SILHOUETTE_PATH}"
    )

cluster_consensus_raw = pd.read_csv(
    CONSENSUS_PATH
)

cluster_silhouette_raw = pd.read_csv(
    SILHOUETTE_PATH
)

require_columns(
    cluster_consensus_raw,
    consensus_columns,
    "Consensus summary",
)

require_columns(
    cluster_silhouette_raw,
    silhouette_columns,
    "Silhouette summary",
)


# =============================================================================
# 3. PREPARE CONSENSUS SUMMARY
# =============================================================================

def prepare_consensus_summary(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Return one consensus-summary row per cluster.

    Exact duplicate exports are removed. Conflicting duplicate cluster rows
    are rejected rather than silently averaged.
    """
    out = df[consensus_columns].copy()

    out["cluster"] = pd.to_numeric(
        out["cluster"],
        errors="coerce",
    )

    out = out.dropna(
        subset=["cluster"]
    )

    out["cluster"] = out["cluster"].astype(int)

    for col in consensus_columns:
        if col != "cluster":
            out[col] = pd.to_numeric(
                out[col],
                errors="coerce",
            )

    out = out.drop_duplicates()

    duplicate_clusters = (
        out.loc[
            out["cluster"].duplicated(
                keep=False
            ),
            "cluster",
        ]
        .unique()
        .tolist()
    )

    if duplicate_clusters:
        raise ValueError(
            "The consensus table contains conflicting "
            "duplicate rows for clusters: "
            f"{duplicate_clusters}. "
            "Inspect consensus_cluster_summary.csv "
            "instead of averaging different results."
        )

    return (
        out.sort_values("cluster")
        .reset_index(drop=True)
    )


cluster_consensus = prepare_consensus_summary(
    cluster_consensus_raw
)


# =============================================================================
# 4. PREPARE BASELINE-LINKAGE SILHOUETTE SUMMARY
# =============================================================================

def prepare_baseline_silhouette(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, str]:
    """
    Select the silhouette statistics corresponding to the baseline linkage.

    Priority
    --------
    1. is_baseline_linkage == True
    2. linkage_method == "ward"
    3. Reject the table if the baseline cannot be identified

    The four linkage methods must not be collapsed by median because they
    describe different clustering solutions.
    """
    out = df.copy()

    out["cluster"] = pd.to_numeric(
        out["cluster"],
        errors="coerce",
    )

    out = out.dropna(
        subset=["cluster"]
    )

    out["cluster"] = out["cluster"].astype(int)

    baseline_label = None

    if "is_baseline_linkage" in out.columns:
        baseline_flag = (
            out["is_baseline_linkage"]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(
                [
                    "true",
                    "1",
                    "yes",
                ]
            )
        )

        if baseline_flag.any():
            out = out.loc[
                baseline_flag
            ].copy()

            baseline_label = (
                str(
                    out["linkage_method"].iloc[0]
                )
                if "linkage_method" in out.columns
                else "baseline"
            )

    if baseline_label is None:
        if "linkage_method" not in out.columns:
            raise ValueError(
                "The silhouette file contains multiple "
                "rows per cluster but has neither "
                "`is_baseline_linkage` nor "
                "`linkage_method`."
            )

        linkage_normalized = (
            out["linkage_method"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        ward_mask = linkage_normalized.eq("ward")

        if not ward_mask.any():
            available = sorted(
                linkage_normalized.unique().tolist()
            )

            raise ValueError(
                "Could not identify the baseline linkage. "
                f"Available methods: {available}"
            )

        out = out.loc[
            ward_mask
        ].copy()

        baseline_label = "ward"

    out = out[
        silhouette_columns
    ].copy()

    for col in silhouette_columns:
        if col != "cluster":
            out[col] = pd.to_numeric(
                out[col],
                errors="coerce",
            )

    out = out.drop_duplicates()

    duplicate_clusters = (
        out.loc[
            out["cluster"].duplicated(
                keep=False
            ),
            "cluster",
        ]
        .unique()
        .tolist()
    )

    if duplicate_clusters:
        raise ValueError(
            "The baseline-linkage silhouette rows "
            "still contain duplicate clusters: "
            f"{duplicate_clusters}"
        )

    return (
        out.sort_values("cluster")
        .reset_index(drop=True),
        baseline_label,
    )


cluster_silhouette, baseline_linkage = (
    prepare_baseline_silhouette(
        cluster_silhouette_raw
    )
)

print(
    "Consensus summary file:",
    CONSENSUS_PATH.name,
)

print(
    "Silhouette summary file:",
    SILHOUETTE_PATH.name,
)

print(
    "Silhouette statistics used:",
    f"{baseline_linkage} linkage",
)


# =============================================================================
# 5. MERGE ROBUSTNESS STATISTICS
# =============================================================================

cluster_quality = cluster_consensus.merge(
    cluster_silhouette,
    on="cluster",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

unmatched = cluster_quality.loc[
    cluster_quality["_merge"] != "both"
]

if not unmatched.empty:
    print(
        "\nWarning: some clusters were not present "
        "in both robustness tables:"
    )

    print(
        unmatched[
            [
                "cluster",
                "_merge",
            ]
        ].to_string(index=False)
    )

cluster_quality = (
    cluster_quality
    .drop(columns="_merge")
    .sort_values("cluster")
    .reset_index(drop=True)
)

cluster_quality.to_csv(
    VIS_DIR / "all_cluster_quality_summary.csv",
    index=False,
)

print("\nCluster-level robustness statistics:")
print(
    cluster_quality.to_string(index=False)
)


# =============================================================================
# 6. FIXED CLUSTER LIST — NO AUTOMATIC SELECTION
# =============================================================================

# Every cluster appearing in the final labels and normalized panel is retained.
all_clusters = sorted(
    labels_table["cluster"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

if not all_clusters:
    raise RuntimeError(
        "No cluster labels were found."
    )

print(
    "\nClusters that will be visualized:",
    all_clusters,
)

quality_lookup = (
    cluster_quality
    .set_index("cluster")
)


# =============================================================================
# 7. MEMBER RANKING AND ROBUST REPRESENTATIVE
# =============================================================================

def minmax_score(
    series: pd.Series,
) -> pd.Series:
    """Map a numeric series to the interval [0, 1]."""
    x = pd.to_numeric(
        series,
        errors="coerce",
    )

    if x.notna().sum() == 0:
        return pd.Series(
            0.5,
            index=x.index,
        )

    low = x.min()
    high = x.max()

    if np.isclose(low, high):
        return pd.Series(
            0.5,
            index=x.index,
        )

    return (x - low) / (high - low)


def available_cluster_members(
    cluster: int,
    labels: pd.DataFrame,
    panel: pd.DataFrame,
) -> list[str]:
    """
    Return members of one cluster that are available in the panel.
    """
    terms = (
        labels.index[
            labels["cluster"] == cluster
        ]
        .astype(str)
        .tolist()
    )

    return [
        term
        for term in terms
        if term in panel.columns
    ]


def rank_cluster_members(
    cluster: int,
    labels: pd.DataFrame,
    panel: pd.DataFrame,
) -> pd.DataFrame:
    """
    Rank member terms by representativeness.

    Ranking components
    ------------------
    - term-level stability, when available;
    - correlation with the preliminary cluster median;
    - data coverage.
    """
    members = available_cluster_members(
        cluster=cluster,
        labels=labels,
        panel=panel,
    )

    if not members:
        return pd.DataFrame(
            columns=[
                "term",
                "cluster",
                "stability_score",
                "centroid_correlation",
                "coverage",
                "representative_score",
            ]
        )

    sub = panel[members]

    preliminary_centroid = (
        sub.median(
            axis=1,
            skipna=True,
        )
    )

    rows = []

    for term in members:
        term_series = sub[term]

        valid = pd.concat(
            [
                term_series.rename("term"),
                preliminary_centroid.rename(
                    "centroid"
                ),
            ],
            axis=1,
        ).dropna()

        correlation = (
            valid["term"].corr(
                valid["centroid"]
            )
            if len(valid) >= 20
            else np.nan
        )

        stability = (
            labels.at[
                term,
                "stability_score",
            ]
            if "stability_score" in labels.columns
            else np.nan
        )

        rows.append(
            {
                "term": term,
                "cluster": cluster,
                "stability_score": stability,
                "centroid_correlation": correlation,
                "coverage": float(
                    term_series.notna().mean()
                ),
            }
        )

    ranked = pd.DataFrame(rows)

    correlation_fill = (
        ranked["centroid_correlation"]
        .median()
    )

    if pd.isna(correlation_fill):
        correlation_fill = 0.0

    correlation_component = (
        ranked["centroid_correlation"]
        .fillna(correlation_fill)
        .clip(-1, 1)
        .add(1)
        .div(2)
    )

    coverage_component = (
        ranked["coverage"]
        .fillna(0)
        .clip(0, 1)
    )

    if ranked["stability_score"].notna().any():
        stability_fill = (
            ranked["stability_score"]
            .median()
        )

        stability_component = minmax_score(
            ranked["stability_score"]
            .fillna(stability_fill)
        )

        ranked["representative_score"] = (
            0.55 * stability_component
            + 0.35 * correlation_component
            + 0.10 * coverage_component
        )
    else:
        ranked["representative_score"] = (
            0.85 * correlation_component
            + 0.15 * coverage_component
        )

    return (
        ranked.sort_values(
            [
                "representative_score",
                "centroid_correlation",
                "coverage",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )


def robust_centroid(
    panel: pd.DataFrame,
    ranked_terms: pd.DataFrame,
    max_terms: int,
) -> tuple[pd.Series, list[str]]:
    """
    Construct the cluster representative from its most representative terms.

    A pointwise median limits the influence of isolated extreme spikes.
    """
    selected_terms = (
        ranked_terms
        .head(max_terms)["term"]
        .astype(str)
        .tolist()
    )

    if not selected_terms:
        raise ValueError(
            "No terms are available for centroid construction."
        )

    centroid = (
        panel[selected_terms]
        .median(
            axis=1,
            skipna=True,
        )
        .interpolate(
            limit_direction="both"
        )
    )

    centroid.name = "cluster_centroid"

    return centroid, selected_terms


# =============================================================================
# 8. DISPLAY SMOOTHING
# =============================================================================

def choose_smoothing_window(
    series: pd.Series,
    config: ClusterShapeConfig,
) -> int:
    """
    Choose a bounded, odd smoothing window for visualization only.
    """
    n = int(
        series.notna().sum()
    )

    proposed = int(
        round(
            np.sqrt(
                max(n, 1)
            ) / 1.8
        )
    )

    proposed = max(
        config.min_smooth_days,
        proposed,
    )

    proposed = min(
        config.max_smooth_days,
        proposed,
    )

    if proposed % 2 == 0:
        proposed += 1

    if proposed > config.max_smooth_days:
        proposed -= 2

    return max(
        5,
        proposed,
    )


def smooth_for_display(
    series: pd.Series,
    window: int,
) -> pd.Series:
    """
    Smooth a series using Savitzky-Golay filtering.

    The raw centroid remains unchanged for diagnostics and exports.
    """
    x = (
        series
        .interpolate(
            limit_direction="both"
        )
        .astype(float)
    )

    if len(x) < 7:
        return x

    usable_window = min(
        window,
        len(x) - 1,
    )

    if usable_window % 2 == 0:
        usable_window -= 1

    usable_window = max(
        5,
        usable_window,
    )

    if usable_window >= len(x):
        return x

    smoothed = savgol_filter(
        x.to_numpy(),
        window_length=usable_window,
        polyorder=min(
            3,
            usable_window - 2,
        ),
        mode="interp",
    )

    return pd.Series(
        smoothed,
        index=x.index,
        name=x.name,
    )


# =============================================================================
# 9. HIGH-INFORMATION LOCAL WINDOW
# =============================================================================

def select_high_information_window(
    centroid: pd.Series,
    window_days: int,
    step_days: int,
) -> tuple[pd.Timestamp, pd.Timestamp, float]:
    """
    Select the local interval that most clearly expresses the trajectory.

    The score rewards:
    - local variation;
    - cumulative movement;
    - upper-tail prominence.

    This controls only the secondary zoom panel. It does not affect the
    underlying clustering or centroid.
    """
    x = (
        centroid
        .dropna()
        .sort_index()
    )

    if x.empty:
        raise ValueError(
            "Cannot select a zoom window from an empty series."
        )

    full_span_days = (
        x.index.max()
        - x.index.min()
    ).days

    if full_span_days <= window_days:
        return (
            x.index.min(),
            x.index.max(),
            np.nan,
        )

    latest_start = (
        x.index.max()
        - pd.Timedelta(
            days=window_days
        )
    )

    starts = pd.date_range(
        start=x.index.min(),
        end=latest_start,
        freq=f"{step_days}D",
    )

    candidates = []

    for start in starts:
        end = (
            start
            + pd.Timedelta(
                days=window_days
            )
        )

        window = (
            x.loc[start:end]
            .dropna()
        )

        required_observations = max(
            30,
            int(
                window_days * 0.60
            ),
        )

        if len(window) < required_observations:
            continue

        variation = float(
            window.std(ddof=0)
        )

        movement = float(
            window.diff()
            .abs()
            .mean()
        )

        prominence = float(
            np.nanpercentile(
                window,
                95,
            )
            - np.nanpercentile(
                window,
                50,
            )
        )

        score = (
            variation
            + 0.75 * movement
            + 0.50 * prominence
        )

        candidates.append(
            (
                score,
                start,
                end,
            )
        )

    if not candidates:
        end = x.index.max()

        start = max(
            x.index.min(),
            end
            - pd.Timedelta(
                days=window_days
            ),
        )

        return (
            start,
            end,
            np.nan,
        )

    score, start, end = max(
        candidates,
        key=lambda item: item[0],
    )

    return (
        start,
        end,
        float(score),
    )


# =============================================================================
# 10. PERIODICITY AND STL
# =============================================================================

def periodicity_diagnostics(
    series: pd.Series,
    config: ClusterShapeConfig,
) -> dict:
    """
    Estimate a dominant period and assess whether it supports STL.

    STL eligibility does not determine whether the cluster is plotted.
    Every cluster is plotted regardless of this result.
    """
    x = (
        series
        .interpolate(
            limit_direction="both"
        )
        .astype(float)
        .to_numpy()
    )

    x = x - np.nanmean(x)

    frequencies, power = periodogram(
        x,
        fs=1.0,
    )

    positive_frequency = (
        frequencies > 0
    )

    periods = np.full_like(
        frequencies,
        fill_value=np.nan,
        dtype=float,
    )

    periods[positive_frequency] = (
        1.0
        / frequencies[positive_frequency]
    )

    period_mask = (
        positive_frequency
        & (
            periods
            >= config.min_period_days
        )
        & (
            periods
            <= config.max_period_days
        )
    )

    if (
        not period_mask.any()
        or np.nansum(
            power[period_mask]
        ) <= 0
    ):
        return {
            "dominant_period_days": np.nan,
            "spectral_concentration": 0.0,
            "n_observed_cycles": 0.0,
            "stl_eligible": False,
        }

    band_power = power[
        period_mask
    ]

    band_periods = periods[
        period_mask
    ]

    peak_position = int(
        np.nanargmax(
            band_power
        )
    )

    dominant_period = int(
        round(
            band_periods[
                peak_position
            ]
        )
    )

    spectral_concentration = float(
        band_power[peak_position]
        / (
            np.nanmean(
                band_power
            )
            + 1e-12
        )
    )

    n_observed_cycles = (
        len(x)
        / max(
            dominant_period,
            1,
        )
    )

    stl_eligible = bool(
        spectral_concentration
        >= config.min_spectral_concentration
        and n_observed_cycles
        >= config.min_cycles_for_stl
    )

    return {
        "dominant_period_days": dominant_period,
        "spectral_concentration": spectral_concentration,
        "n_observed_cycles": float(
            n_observed_cycles
        ),
        "stl_eligible": stl_eligible,
    }


def fit_stl_if_supported(
    centroid: pd.Series,
    periodicity: dict,
):
    """
    Fit robust STL when the periodicity evidence is sufficiently strong.
    """
    if not periodicity[
        "stl_eligible"
    ]:
        return None

    period = int(
        periodicity[
            "dominant_period_days"
        ]
    )

    if (
        period < 3
        or len(
            centroid.dropna()
        ) < 2 * period
    ):
        return None

    x = (
        centroid
        .interpolate(
            limit_direction="both"
        )
        .astype(float)
    )

    try:
        return STL(
            x,
            period=period,
            robust=True,
        ).fit()

    except Exception as error:
        print(
            "STL fit failed:",
            str(error),
        )

        return None


def component_strength(
    component: pd.Series | np.ndarray,
    residual: pd.Series | np.ndarray,
) -> float:
    """
    Calculate component strength on the interval [0, 1].
    """
    component_array = np.asarray(
        component,
        dtype=float,
    )

    residual_array = np.asarray(
        residual,
        dtype=float,
    )

    denominator = np.nanvar(
        component_array
        + residual_array
    )

    if denominator <= 1e-12:
        return 0.0

    strength = (
        1.0
        - np.nanvar(
            residual_array
        )
        / denominator
    )

    return float(
        np.clip(
            strength,
            0.0,
            1.0,
        )
    )


# =============================================================================
# 11. PLAIN-LANGUAGE SHAPE DESCRIPTION
# =============================================================================

def robust_spike_score(
    series: pd.Series,
) -> float:
    """
    Measure the upper-tail peak relative to the series' robust scale.
    """
    x = (
        series
        .dropna()
        .astype(float)
    )

    if x.empty:
        return np.nan

    median = float(
        x.median()
    )

    mad = float(
        (
            x - median
        )
        .abs()
        .median()
    )

    if mad <= 1e-12:
        return 0.0

    upper = float(
        x.quantile(0.99)
    )

    return (
        0.6745
        * (
            upper - median
        )
        / mad
    )


def trend_direction_score(
    series: pd.Series,
) -> float:
    """
    Compare the beginning and end of the series relative to its variability.
    """
    x = (
        series
        .dropna()
        .astype(float)
    )

    if len(x) < 20:
        return 0.0

    block = max(
        5,
        len(x) // 10,
    )

    starting_level = float(
        x.iloc[:block]
        .median()
    )

    ending_level = float(
        x.iloc[-block:]
        .median()
    )

    scale = float(
        x.std(ddof=0)
    )

    if scale <= 1e-12:
        return 0.0

    return (
        ending_level
        - starting_level
    ) / scale


def describe_shape(
    centroid: pd.Series,
    periodicity: dict,
    stl_result,
) -> dict:
    """
    Generate an accessible descriptive label for one cluster.

    The label is a summary of the centroid, not a replacement for manual
    substantive interpretation.
    """
    spike_score = robust_spike_score(
        centroid
    )

    trend_score = trend_direction_score(
        centroid
    )

    if stl_result is not None:
        trend_strength = component_strength(
            stl_result.trend,
            stl_result.resid,
        )

        seasonal_strength = component_strength(
            stl_result.seasonal,
            stl_result.resid,
        )
    else:
        trend_strength = np.nan
        seasonal_strength = np.nan

    if (
        periodicity["stl_eligible"]
        and pd.notna(
            seasonal_strength
        )
        and seasonal_strength >= 0.45
    ):
        shape_label = (
            "Recurring seasonal pulses"
        )

        lay_description = (
            "Search attention repeatedly rises and falls "
            "in a recognizable cycle."
        )

    elif (
        pd.notna(spike_score)
        and spike_score >= 5.0
    ):
        shape_label = (
            "Intermittent attention surges"
        )

        lay_description = (
            "Attention usually remains near its normal "
            "level but occasionally jumps sharply."
        )

    elif trend_score >= 0.75:
        shape_label = (
            "Gradual attention climb"
        )

        lay_description = (
            "Search attention becomes progressively "
            "stronger over time."
        )

    elif trend_score <= -0.75:
        shape_label = (
            "Gradual attention decline"
        )

        lay_description = (
            "Search attention becomes progressively "
            "weaker over time."
        )

    elif periodicity["stl_eligible"]:
        shape_label = (
            "Repeated attention cycle"
        )

        lay_description = (
            "The trajectory contains recurring movement, "
            "although seasonality is not dominant."
        )

    else:
        shape_label = (
            "Irregular event-driven movement"
        )

        lay_description = (
            "The trajectory contains notable attention "
            "episodes without a stable repeating cycle."
        )

    return {
        "shape_label": shape_label,
        "lay_description": lay_description,
        "spike_score": spike_score,
        "trend_direction_score": trend_score,
        "trend_strength": trend_strength,
        "seasonal_strength": seasonal_strength,
    }


# =============================================================================
# 12. BUILD REPRESENTATION FOR EVERY CLUSTER
# =============================================================================

cluster_objects = {}
representative_rows = []
summary_rows = []

for cluster in all_clusters:
    ranked_terms = rank_cluster_members(
        cluster=cluster,
        labels=labels_table,
        panel=panel_clean,
    )

    if ranked_terms.empty:
        print(
            f"Cluster {cluster}: no member terms "
            "were found in panel_norm. Skipping."
        )

        continue

    centroid_raw, centroid_terms = robust_centroid(
        panel=panel_clean,
        ranked_terms=ranked_terms,
        max_terms=CFG.max_terms_for_centroid,
    )

    smooth_window = choose_smoothing_window(
        centroid_raw,
        CFG,
    )

    centroid_display = smooth_for_display(
        centroid_raw,
        window=smooth_window,
    )

    zoom_start, zoom_end, zoom_score = (
        select_high_information_window(
            centroid=centroid_display,
            window_days=CFG.zoom_window_days,
            step_days=CFG.zoom_step_days,
        )
    )

    periodicity = periodicity_diagnostics(
        centroid_raw,
        CFG,
    )

    stl_result = fit_stl_if_supported(
        centroid_raw,
        periodicity,
    )

    shape_description = describe_shape(
        centroid=centroid_raw,
        periodicity=periodicity,
        stl_result=stl_result,
    )

    representative_terms = (
        ranked_terms
        .head(
            CFG.n_representative_terms
        )["term"]
        .astype(str)
        .tolist()
    )

    cluster_objects[cluster] = {
        "ranked_terms": ranked_terms,
        "centroid_terms": centroid_terms,
        "representative_terms": representative_terms,
        "centroid_raw": centroid_raw,
        "centroid_display": centroid_display,
        "smooth_window": smooth_window,
        "zoom_start": zoom_start,
        "zoom_end": zoom_end,
        "zoom_score": zoom_score,
        "periodicity": periodicity,
        "stl_result": stl_result,
        "description": shape_description,
    }

    for rank, row in ranked_terms.iterrows():
        representative_rows.append(
            {
                "cluster": cluster,
                "term": row["term"],
                "rank_within_cluster": rank + 1,
                "used_in_centroid": (
                    row["term"]
                    in centroid_terms
                ),
                "shown_on_figure": (
                    row["term"]
                    in representative_terms
                ),
                "stability_score": row.get(
                    "stability_score",
                    np.nan,
                ),
                "centroid_correlation": row[
                    "centroid_correlation"
                ],
                "coverage": row["coverage"],
                "representative_score": row[
                    "representative_score"
                ],
            }
        )

    if cluster in quality_lookup.index:
        quality = quality_lookup.loc[
            cluster
        ]
    else:
        quality = pd.Series(
            dtype=float
        )

    summary_rows.append(
        {
            "cluster": cluster,
            "n_terms_in_labels": int(
                (
                    labels_table["cluster"]
                    == cluster
                ).sum()
            ),
            "n_terms_in_panel": len(
                ranked_terms
            ),
            "n_terms_used_in_centroid": len(
                centroid_terms
            ),
            "mean_within_consensus": quality.get(
                "mean_within_consensus",
                np.nan,
            ),
            "p10_within_consensus": quality.get(
                "p10_within_consensus",
                np.nan,
            ),
            "consensus_margin": quality.get(
                "consensus_margin",
                np.nan,
            ),
            "silhouette_mean": quality.get(
                "silhouette_mean",
                np.nan,
            ),
            "silhouette_median": quality.get(
                "silhouette_median",
                np.nan,
            ),
            "negative_silhouette_share": quality.get(
                "negative_silhouette_share",
                np.nan,
            ),
            "shape_label": shape_description[
                "shape_label"
            ],
            "lay_description": shape_description[
                "lay_description"
            ],
            "spike_score": shape_description[
                "spike_score"
            ],
            "trend_direction_score": shape_description[
                "trend_direction_score"
            ],
            "trend_strength": shape_description[
                "trend_strength"
            ],
            "seasonal_strength": shape_description[
                "seasonal_strength"
            ],
            "dominant_period_days": periodicity[
                "dominant_period_days"
            ],
            "spectral_concentration": periodicity[
                "spectral_concentration"
            ],
            "n_observed_cycles": periodicity[
                "n_observed_cycles"
            ],
            "stl_eligible": periodicity[
                "stl_eligible"
            ],
            "smoothing_window_days": smooth_window,
            "zoom_start": zoom_start,
            "zoom_end": zoom_end,
            "zoom_information_score": zoom_score,
            "representative_terms": " | ".join(
                representative_terms
            ),
        }
    )


representatives_df = pd.DataFrame(
    representative_rows
)

shape_summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    "cluster"
)

representatives_df.to_csv(
    VIS_DIR
    / "all_cluster_representatives.csv",
    index=False,
)

shape_summary_df.to_csv(
    VIS_DIR
    / "all_cluster_shape_summary.csv",
    index=False,
)


# =============================================================================
# 13. FIGURE HELPERS
# =============================================================================

def humanize_term(
    term: str,
) -> str:
    """Convert a stored term name into a readable label."""
    return (
        str(term)
        .replace("_", " ")
        .strip()
        .title()
    )


def style_axis(
    ax: plt.Axes,
) -> None:
    """Apply restrained publication-style formatting."""
    ax.spines["top"].set_visible(
        False
    )

    ax.spines["right"].set_visible(
        False
    )

    ax.grid(
        axis="y",
        alpha=0.18,
        linewidth=0.7,
    )

    ax.tick_params(
        axis="both",
        labelsize=8,
    )


def quality_text(
    cluster: int,
    quality_lookup: pd.DataFrame,
) -> str:
    """
    Create a concise cluster-quality subtitle.
    """
    if cluster not in quality_lookup.index:
        return (
            "Robustness statistics unavailable"
        )

    row = quality_lookup.loc[
        cluster
    ]

    n_terms = row.get(
        "n_terms",
        np.nan,
    )

    consensus = row.get(
        "mean_within_consensus",
        np.nan,
    )

    silhouette = row.get(
        "silhouette_mean",
        np.nan,
    )

    parts = []

    if pd.notna(n_terms):
        parts.append(
            f"n={int(n_terms)} terms"
        )

    if pd.notna(consensus):
        parts.append(
            f"consensus={consensus:.2f}"
        )

    if pd.notna(silhouette):
        parts.append(
            f"silhouette={silhouette:.2f}"
        )

    return " · ".join(parts)


# =============================================================================
# 14. ONE STANDALONE FIGURE PER CLUSTER
# =============================================================================

for cluster in sorted(
    cluster_objects
):
    obj = cluster_objects[
        cluster
    ]

    centroid = obj[
        "centroid_display"
    ]

    representative_terms = obj[
        "representative_terms"
    ]

    smooth_window = obj[
        "smooth_window"
    ]

    zoom_start = obj[
        "zoom_start"
    ]

    zoom_end = obj[
        "zoom_end"
    ]

    description = obj[
        "description"
    ]

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(
            CFG.figure_width,
            CFG.figure_height,
        ),
        squeeze=False,
    )

    ax_full = axes[0, 0]
    ax_zoom = axes[0, 1]

    # -------------------------------------------------------------------------
    # Left panel: full cluster history
    # -------------------------------------------------------------------------
    ax_full.plot(
        centroid.index,
        centroid,
        linewidth=2.6,
        label="Robust cluster shape",
        zorder=4,
    )

    ax_full.axhline(
        0,
        linewidth=0.8,
        alpha=0.35,
        zorder=1,
    )

    ax_full.axvspan(
        zoom_start,
        zoom_end,
        alpha=0.08,
        linewidth=0,
        zorder=0,
    )

    ax_full.set_title(
        "Full attention trajectory",
        loc="left",
        fontsize=11,
        fontweight="bold",
    )

    ax_full.set_xlabel(
        "Date",
        fontsize=9,
    )

    ax_full.set_ylabel(
        "Normalized search attention",
        fontsize=9,
    )

    style_axis(
        ax_full
    )

    # -------------------------------------------------------------------------
    # Right panel: informative local interval
    # -------------------------------------------------------------------------
    centroid_zoom = centroid.loc[
        zoom_start:zoom_end
    ]

    for term in representative_terms:
        term_display = smooth_for_display(
            panel_clean[term],
            smooth_window,
        ).loc[
            zoom_start:zoom_end
        ]

        ax_zoom.plot(
            term_display.index,
            term_display,
            linewidth=1.1,
            alpha=0.45,
            label=humanize_term(
                term
            ),
            zorder=2,
        )

    ax_zoom.plot(
        centroid_zoom.index,
        centroid_zoom,
        linewidth=2.8,
        label="Robust cluster shape",
        zorder=5,
    )

    ax_zoom.axhline(
        0,
        linewidth=0.8,
        alpha=0.35,
        zorder=1,
    )

    ax_zoom.set_title(
        (
            "Representative local shape\n"
            f"{zoom_start:%b %Y}–"
            f"{zoom_end:%b %Y}"
        ),
        loc="left",
        fontsize=11,
        fontweight="bold",
    )

    ax_zoom.set_xlabel(
        "Date",
        fontsize=9,
    )

    ax_zoom.set_ylabel(
        "Normalized search attention",
        fontsize=9,
    )

    ax_zoom.text(
        0.01,
        0.98,
        description[
            "lay_description"
        ],
        transform=ax_zoom.transAxes,
        va="top",
        ha="left",
        fontsize=8.5,
        linespacing=1.3,
        bbox={
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.82,
            "pad": 3,
        },
    )

    style_axis(
        ax_zoom
    )

    handles, labels = (
        ax_zoom.get_legend_handles_labels()
    )

    if "Robust cluster shape" in labels:
        centroid_position = labels.index(
            "Robust cluster shape"
        )

        legend_order = [
            centroid_position
        ] + [
            i
            for i in range(
                len(labels)
            )
            if i != centroid_position
        ]
    else:
        legend_order = list(
            range(
                len(labels)
            )
        )

    ax_zoom.legend(
        [
            handles[i]
            for i in legend_order
        ],
        [
            labels[i]
            for i in legend_order
        ],
        loc="lower left",
        fontsize=7.5,
        frameon=False,
        ncol=2,
    )

    figure_title = (
        f"Cluster {cluster}: "
        f"{description['shape_label']}"
    )

    figure_subtitle = quality_text(
        cluster,
        quality_lookup,
    )

    fig.suptitle(
        (
            f"{figure_title}\n"
            f"{figure_subtitle} · "
            f"display smoothing="
            f"{smooth_window} days"
        ),
        fontsize=14,
        fontweight="bold",
        y=1.02,
    )

    fig.tight_layout()

    figure_path = (
        CLUSTER_FIG_DIR
        / f"cluster_{cluster}_shape.png"
    )

    fig.savefig(
        figure_path,
        dpi=CFG.dpi,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    print(
        f"Saved cluster {cluster}: "
        f"{figure_path.name}"
    )


# =============================================================================
# 15. STL DECOMPOSITION — ONE FIGURE PER ELIGIBLE CLUSTER
# =============================================================================

def plot_stl_decomposition(
    cluster: int,
    centroid: pd.Series,
    stl_result,
    periodicity: dict,
    description: dict,
    output_path: Path,
    dpi: int,
) -> None:
    """
    Plot a standalone STL decomposition for one cluster.
    """
    fig, axes = plt.subplots(
        nrows=4,
        ncols=1,
        figsize=(
            CFG.stl_figure_width,
            CFG.stl_figure_height,
        ),
        sharex=True,
    )

    components = [
        (
            "Observed cluster shape",
            centroid,
        ),
        (
            "Long-run movement",
            stl_result.trend,
        ),
        (
            (
                "Recurring component "
                f"({int(periodicity['dominant_period_days'])}"
                "-day period)"
            ),
            stl_result.seasonal,
        ),
        (
            "Irregular remainder",
            stl_result.resid,
        ),
    ]

    for ax, (
        component_title,
        component_values,
    ) in zip(
        axes,
        components,
    ):
        ax.plot(
            centroid.index,
            component_values,
            linewidth=1.8,
        )

        ax.axhline(
            0,
            linewidth=0.7,
            alpha=0.30,
        )

        ax.set_title(
            component_title,
            loc="left",
            fontsize=10,
            fontweight="bold",
        )

        style_axis(
            ax
        )

    axes[-1].set_xlabel(
        "Date",
        fontsize=9,
    )

    fig.suptitle(
        (
            f"Cluster {cluster}: "
            f"{description['shape_label']}\n"
            "Robust STL decomposition · "
            "spectral concentration="
            f"{periodicity['spectral_concentration']:.1f}"
        ),
        fontsize=14,
        fontweight="bold",
    )

    fig.tight_layout(
        rect=[
            0,
            0,
            1,
            0.94,
        ]
    )

    fig.savefig(
        output_path,
        dpi=dpi,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


for cluster in sorted(
    cluster_objects
):
    obj = cluster_objects[
        cluster
    ]

    if obj["stl_result"] is None:
        print(
            f"Cluster {cluster}: main shape figure saved; "
            "STL omitted because periodicity was not "
            "sufficiently supported."
        )

        continue

    stl_path = (
        STL_DIR
        / f"cluster_{cluster}_stl.png"
    )

    plot_stl_decomposition(
        cluster=cluster,
        centroid=obj[
            "centroid_raw"
        ],
        stl_result=obj[
            "stl_result"
        ],
        periodicity=obj[
            "periodicity"
        ],
        description=obj[
            "description"
        ],
        output_path=stl_path,
        dpi=CFG.dpi,
    )


# =============================================================================
# 16. EXPORT SHAPE DICTIONARY
# =============================================================================

shape_dictionary_columns = [
    "cluster",
    "n_terms_in_labels",
    "n_terms_in_panel",
    "mean_within_consensus",
    "p10_within_consensus",
    "consensus_margin",
    "silhouette_mean",
    "silhouette_median",
    "negative_silhouette_share",
    "shape_label",
    "lay_description",
    "representative_terms",
    "dominant_period_days",
    "spectral_concentration",
    "stl_eligible",
    "trend_strength",
    "seasonal_strength",
    "spike_score",
]

cluster_shape_dictionary = (
    shape_summary_df[
        shape_dictionary_columns
    ]
    .sort_values("cluster")
)

cluster_shape_dictionary.to_csv(
    VIS_DIR
    / "all_cluster_shape_dictionary.csv",
    index=False,
)

print("\nAll-cluster shape dictionary:")
print(
    cluster_shape_dictionary[
        [
            "cluster",
            "n_terms_in_labels",
            "shape_label",
            "representative_terms",
            "mean_within_consensus",
            "silhouette_mean",
            "stl_eligible",
        ]
    ].to_string(index=False)
)

print(
    "\nAll cluster visualizations saved to:"
)

print(
    CLUSTER_FIG_DIR
)

print(
    "\nSTL decompositions saved to:"
)

print(
    STL_DIR
)

Consensus summary file: consensus_cluster_summary.csv
Silhouette summary file: linkage_cluster_silhouette.csv
Silhouette statistics used: ward linkage

Cluster-level robustness statistics:
 cluster  n_terms  mean_within_consensus  p10_within_consensus  max_mean_between_consensus  consensus_margin  silhouette_mean  silhouette_median  silhouette_p10  negative_silhouette_share
       1       90               0.579766              0.043478                    0.141590          0.438176         0.258915           0.294377        0.066660                   0.011111
       2       46               0.749768              0.219512                    0.080306          0.669462         0.096686           0.125028       -0.048445                   0.130435
       3       40               0.819523              0.434701                    0.062117          0.757406         0.064518           0.064942       -0.021230                   0.175000
       4       58               0.761805              0.490

## 8. Volatility forecasting with cluster indices

Primary model: expanding-window HAR-X on log daily realized variance.
Baseline: HAR(1, 5, 22). Candidate: HAR + one lagged cluster index.
Each cluster is tested independently; p-values are adjusted with Benjamini-Hochberg.

Required notebook objects from Steps 1-6:
  panel_norm, labels_final, term_stability, consensus_cluster_summary,
  linkage_term_silhouette, LINKAGE_METHOD, OUTPUT_DIR, END_DATE, RANDOM_STATE

In [42]:
from __future__ import annotations

import json
import math
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.api import OLS, add_constant


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

@dataclass(frozen=True)
class VolatilityForecastConfig:
    market_path: Path = Path(
            "C:/Python/CSUREMM/data/shock_detection/SP500_data.csv"
    )
    output_dir: Path = Path(OUTPUT_DIR) / "08_prediction"
    date_column_candidates: tuple[str, ...] = (
        "date", "time", "datetime", "timestamp"
    )
    close_column_candidates: tuple[str, ...] = (
        "adj close", "adjusted close", "close", "price"
    )
    min_term_stability: float = 0.50
    min_term_silhouette: float = 0.00
    min_terms_per_cluster: int = 10
    consensus_margin_floor: float = 0.00
    initial_train_fraction: float = 0.70
    min_initial_train_days: int = 504
    refit_every: int = 5
    forecast_horizons: tuple[int, ...] = (1, 5, 22)
    dm_hac_lags: int = 10
    bootstrap_replications: int = 2000
    bootstrap_block_length: int = 10
    random_state: int = int(globals().get("RANDOM_STATE", 42))
    # "strict_post_discovery" is the publication analysis and requires data
    # after the clustering freeze date. "exploratory_pseudo_oos" runs within
    # the discovery sample and must be labelled exploratory.
    evaluation_mode: str = "exploratory_pseudo_oos"
    discovery_end_date: str = str(globals().get("END_DATE", "2026-05-31"))


CFG = VolatilityForecastConfig()
CFG.output_dir.mkdir(parents=True, exist_ok=True)

if CFG.evaluation_mode not in {
    "strict_post_discovery",
    "exploratory_pseudo_oos",
}:
    raise ValueError(
        "evaluation_mode must be 'strict_post_discovery' or "
        "'exploratory_pseudo_oos'."
    )


# -----------------------------------------------------------------------------
# Validation helpers
# -----------------------------------------------------------------------------

def require_columns(df: pd.DataFrame, columns: Iterable[str], name: str) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")


def normalize_column_name(name: str) -> str:
    return " ".join(str(name).strip().lower().replace("_", " ").split())


def select_column(df: pd.DataFrame, candidates: Iterable[str]) -> str | None:
    normalized = {normalize_column_name(c): c for c in df.columns}
    for candidate in candidates:
        key = normalize_column_name(candidate)
        if key in normalized:
            return normalized[key]
    return None


def assert_not_future_aligned(feature: pd.Series, target: pd.Series) -> None:
    if not feature.index.equals(target.index):
        raise ValueError("Feature and target indices must be aligned before modeling.")
    if feature.index.has_duplicates:
        raise ValueError("Modeling index contains duplicate dates.")


# -----------------------------------------------------------------------------
# Market loading and daily variance proxy
# -----------------------------------------------------------------------------

def load_market_variance(path: Path) -> pd.DataFrame:
    """
    Load daily market data and construct a variance proxy.

    Preferred proxy: Rogers-Satchell variance using OHLC data, which is robust
    to non-zero drift. Fallback: squared close-to-close log return.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Market file not found: {path.resolve()}\n"
            "Set SP500_PATH before Step 8 or edit CFG.market_path."
        )

    raw = pd.read_csv(path)
    if raw.empty:
        raise ValueError(f"Market file is empty: {path}")

    date_col = select_column(raw, CFG.date_column_candidates)
    if date_col is None:
        date_col = raw.columns[0]

    dates = pd.to_datetime(raw[date_col], errors="coerce")
    market = raw.loc[dates.notna()].copy()
    market.index = dates.loc[dates.notna()]
    market = market[~market.index.duplicated(keep="last")].sort_index()

    normalized_map = {normalize_column_name(c): c for c in market.columns}
    ohlc_names = {}
    for canonical in ("open", "high", "low", "close"):
        if canonical in normalized_map:
            ohlc_names[canonical] = normalized_map[canonical]
        elif canonical == "close":
            close_col = select_column(market, CFG.close_column_candidates)
            if close_col is not None:
                ohlc_names[canonical] = close_col

    if set(ohlc_names) == {"open", "high", "low", "close"}:
        o = pd.to_numeric(market[ohlc_names["open"]], errors="coerce")
        h = pd.to_numeric(market[ohlc_names["high"]], errors="coerce")
        l = pd.to_numeric(market[ohlc_names["low"]], errors="coerce")
        c = pd.to_numeric(market[ohlc_names["close"]], errors="coerce")

        valid = (o > 0) & (h > 0) & (l > 0) & (c > 0)
        rs = pd.Series(np.nan, index=market.index, dtype=float)
        rs.loc[valid] = (
            np.log(h.loc[valid] / c.loc[valid])
            * np.log(h.loc[valid] / o.loc[valid])
            + np.log(l.loc[valid] / c.loc[valid])
            * np.log(l.loc[valid] / o.loc[valid])
        )
        variance = rs.clip(lower=1e-12)
        proxy_name = "rogers_satchell"
        close = c
    else:
        close_col = select_column(market, CFG.close_column_candidates)
        if close_col is None:
            raise KeyError(
                "Market CSV needs OHLC columns or at least a Close/Adj Close column."
            )
        close = pd.to_numeric(market[close_col], errors="coerce")
        close = close.where(close > 0)
        log_return = np.log(close).diff()
        variance = log_return.pow(2).clip(lower=1e-12)
        proxy_name = "squared_log_return"

    out = pd.DataFrame(
        {
            "close": close,
            "daily_variance": variance,
            "log_variance": np.log(variance),
        }
    ).replace([np.inf, -np.inf], np.nan)

    out.attrs["variance_proxy"] = proxy_name
    return out.dropna(subset=["log_variance"])


# -----------------------------------------------------------------------------
# Step 6 integration and cluster-index construction
# -----------------------------------------------------------------------------

def prepare_term_quality_table() -> pd.DataFrame:
    """
    Merge fixed cluster membership with Step 5/6 term diagnostics.

    Primary term weight:
        term_stability * max(baseline-linkage silhouette, 0)

    Consensus is cluster-level evidence, so it is used as a reliability screen
    rather than multiplied into within-cluster weights (a common cluster scalar
    would cancel after weight normalization).
    """
    if not isinstance(panel_norm, pd.DataFrame):
        raise TypeError("panel_norm must be a pandas DataFrame.")

    labels = pd.Series(labels_final, copy=True)
    labels.index = labels.index.astype(str)
    labels.name = "cluster"

    stability = term_stability.copy()
    require_columns(stability, ["term", "cluster", "term_stability"], "term_stability")
    stability["term"] = stability["term"].astype(str)

    silhouette = linkage_term_silhouette.copy()
    require_columns(
        silhouette,
        ["term", "cluster", "silhouette", "linkage_method"],
        "linkage_term_silhouette",
    )
    silhouette = silhouette.loc[
        silhouette["linkage_method"].astype(str) == str(LINKAGE_METHOD)
    ].copy()
    silhouette["term"] = silhouette["term"].astype(str)

    if silhouette["term"].duplicated().any():
        raise ValueError(
            "Baseline linkage_term_silhouette contains duplicate term rows."
        )

    quality = (
        stability[["term", "cluster", "term_stability", "mindist_margin"]]
        .merge(
            silhouette[["term", "silhouette"]],
            on="term",
            how="inner",
            validate="one_to_one",
        )
    )

    label_check = labels.rename("label_cluster").rename_axis("term").reset_index()
    quality = quality.merge(label_check, on="term", how="inner", validate="one_to_one")
    if not (quality["cluster"].astype(int) == quality["label_cluster"].astype(int)).all():
        raise ValueError("Step 5 stability clusters do not match labels_final.")

    consensus = consensus_cluster_summary.copy()
    require_columns(
        consensus,
        ["cluster", "mean_within_consensus", "consensus_margin"],
        "consensus_cluster_summary",
    )
    if consensus["cluster"].duplicated().any():
        consensus = (
            consensus.groupby("cluster", as_index=False)
            .median(numeric_only=True)
        )

    quality = quality.merge(consensus, on="cluster", how="left", validate="many_to_one")
    quality["eligible"] = (
        (quality["term_stability"] >= CFG.min_term_stability)
        & (quality["silhouette"] >= CFG.min_term_silhouette)
        & (quality["consensus_margin"] >= CFG.consensus_margin_floor)
        & quality["term"].isin(panel_norm.columns.astype(str))
    )
    quality["raw_weight"] = (
        quality["term_stability"].clip(lower=0)
        * quality["silhouette"].clip(lower=0)
    )
    quality.loc[~quality["eligible"], "raw_weight"] = 0.0
    return quality.sort_values(["cluster", "raw_weight"], ascending=[True, False])


def construct_cluster_indices(
    panel: pd.DataFrame,
    quality: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Construct one interpretable weighted average of normalized attention per cluster.

    The contemporaneous index at date t uses only Trends observations dated t.
    Forecast models use index.shift(1), so no date-t attention predicts date-t risk.
    """
    aligned_panel = panel.copy()
    aligned_panel.columns = aligned_panel.columns.astype(str)
    aligned_panel.index = pd.to_datetime(aligned_panel.index)
    aligned_panel = aligned_panel.sort_index()

    indices = {}
    audit_rows = []

    for cluster_id, group in quality.groupby("cluster", sort=True):
        available = group.loc[group["term"].isin(aligned_panel.columns)].copy()
        eligible = available.loc[available["raw_weight"] > 0].copy()

        # Deterministic fallback for weak clusters: use the most stable terms,
        # but retain the cluster in the evaluation rather than silently dropping it.
        if len(eligible) < CFG.min_terms_per_cluster:
            eligible = (
                available.sort_values(
                    ["term_stability", "silhouette", "mindist_margin"],
                    ascending=False,
                )
                .head(min(CFG.min_terms_per_cluster, len(available)))
                .copy()
            )
            eligible["raw_weight"] = eligible["term_stability"].clip(lower=1e-8)
            weighting_rule = "stability_fallback"
        else:
            weighting_rule = "stability_x_positive_silhouette"

        if eligible.empty:
            warnings.warn(f"Cluster {cluster_id} has no terms in panel_norm; skipped.")
            continue

        weights = eligible.set_index("term")["raw_weight"].astype(float)
        weights = weights / weights.sum()
        cluster_series = aligned_panel[weights.index].mul(weights, axis=1).sum(axis=1)
        cluster_series.name = f"cluster_{int(cluster_id)}_index"
        indices[cluster_series.name] = cluster_series

        audit_rows.append(
            {
                "cluster": int(cluster_id),
                "index_name": cluster_series.name,
                "n_available_terms": int(len(available)),
                "n_index_terms": int(len(eligible)),
                "weighting_rule": weighting_rule,
                "effective_n_terms": float(1.0 / np.square(weights).sum()),
                "max_term_weight": float(weights.max()),
                "mean_term_stability": float(eligible["term_stability"].mean()),
                "mean_term_silhouette": float(eligible["silhouette"].mean()),
                "mean_within_consensus": float(eligible["mean_within_consensus"].iloc[0]),
                "consensus_margin": float(eligible["consensus_margin"].iloc[0]),
                "terms": ", ".join(weights.index.tolist()),
            }
        )

    if not indices:
        raise RuntimeError("No cluster indices could be constructed.")

    index_panel = pd.DataFrame(indices).sort_index()
    audit = pd.DataFrame(audit_rows).sort_values("cluster").reset_index(drop=True)
    return index_panel, audit


# -----------------------------------------------------------------------------
# HAR-X design matrix
# -----------------------------------------------------------------------------

def forward_mean(series: pd.Series, horizon: int) -> pd.Series:
    """Mean log variance over t+1,...,t+h, aligned at date t."""
    return series.shift(-1).rolling(horizon).mean().shift(-(horizon - 1))


def build_har_dataset(
    market: pd.DataFrame,
    attention_indices: pd.DataFrame,
    horizon: int,
) -> pd.DataFrame:
    merged = market[["log_variance"]].join(attention_indices, how="inner")
    rv = merged["log_variance"]

    data = pd.DataFrame(index=merged.index)
    data["target"] = forward_mean(rv, horizon)
    data["har_d"] = rv
    data["har_w"] = rv.rolling(5, min_periods=5).mean()
    data["har_m"] = rv.rolling(22, min_periods=22).mean()

    # Strict timing: only yesterday's completed attention index is available.
    for column in attention_indices.columns:
        data[column] = merged[column].shift(1)

    return data.replace([np.inf, -np.inf], np.nan)


# -----------------------------------------------------------------------------
# Expanding-window forecasting
# -----------------------------------------------------------------------------

def fit_ols_predict(
    train: pd.DataFrame,
    test_row: pd.DataFrame,
    predictors: list[str],
) -> float:
    clean_train = train.dropna(subset=["target", *predictors])
    if len(clean_train) <= len(predictors) + 20:
        return np.nan

    x_train = add_constant(clean_train[predictors], has_constant="add")
    x_test = add_constant(test_row[predictors], has_constant="add")
    x_test = x_test.reindex(columns=x_train.columns, fill_value=1.0)

    model = OLS(clean_train["target"], x_train, missing="drop").fit()
    return float(model.predict(x_test).iloc[0])


def expanding_forecasts(
    data: pd.DataFrame,
    cluster_predictor: str,
    initial_train_n: int,
    refit_every: int,
) -> pd.DataFrame:
    """Forecast baseline and one cluster-augmented model on identical dates."""
    baseline_predictors = ["har_d", "har_w", "har_m"]
    augmented_predictors = baseline_predictors + [cluster_predictor]
    required = ["target", *augmented_predictors]
    usable = data.dropna(subset=required).copy()

    if len(usable) <= initial_train_n + 30:
        raise ValueError(
            f"Insufficient observations for {cluster_predictor}: "
            f"{len(usable)} usable rows, initial_train_n={initial_train_n}."
        )

    rows = []
    baseline_model = None
    augmented_model = None

    for i in range(initial_train_n, len(usable)):
        train = usable.iloc[:i]
        test = usable.iloc[[i]]

        if baseline_model is None or (i - initial_train_n) % refit_every == 0:
            xb = add_constant(train[baseline_predictors], has_constant="add")
            xa = add_constant(train[augmented_predictors], has_constant="add")
            baseline_model = OLS(train["target"], xb).fit()
            augmented_model = OLS(train["target"], xa).fit()

        xb_test = add_constant(test[baseline_predictors], has_constant="add")
        xb_test = xb_test.reindex(columns=baseline_model.model.exog_names, fill_value=1.0)
        xa_test = add_constant(test[augmented_predictors], has_constant="add")
        xa_test = xa_test.reindex(columns=augmented_model.model.exog_names, fill_value=1.0)

        rows.append(
            {
                "date": test.index[0],
                "actual_log_variance": float(test["target"].iloc[0]),
                "baseline_forecast": float(baseline_model.predict(xb_test).iloc[0]),
                "augmented_forecast": float(augmented_model.predict(xa_test).iloc[0]),
            }
        )

    return pd.DataFrame(rows).set_index("date")


# -----------------------------------------------------------------------------
# Losses and inference
# -----------------------------------------------------------------------------

def qlike_loss(actual_log_var: np.ndarray, forecast_log_var: np.ndarray) -> np.ndarray:
    actual_var = np.exp(np.asarray(actual_log_var, dtype=float))
    forecast_var = np.exp(np.asarray(forecast_log_var, dtype=float))
    forecast_var = np.clip(forecast_var, 1e-12, np.inf)
    return np.log(forecast_var) + actual_var / forecast_var


def newey_west_mean_test(values: np.ndarray, max_lag: int) -> tuple[float, float]:
    """Two-sided HAC test that the mean of values equals zero."""
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 20:
        return np.nan, np.nan

    centered = x - x.mean()
    gamma0 = float(centered @ centered / n)
    long_run_var = gamma0
    lag_cap = min(max_lag, n - 1)
    for lag in range(1, lag_cap + 1):
        weight = 1.0 - lag / (lag_cap + 1.0)
        gamma = float(centered[lag:] @ centered[:-lag] / n)
        long_run_var += 2.0 * weight * gamma

    se = math.sqrt(max(long_run_var, 0.0) / n)
    if se <= 0:
        return np.nan, np.nan
    statistic = float(x.mean() / se)
    p_value = float(2.0 * stats.norm.sf(abs(statistic)))
    return statistic, p_value


def stationary_block_bootstrap_pvalue(
    loss_diff: np.ndarray,
    replications: int,
    block_length: int,
    seed: int,
) -> float:
    """One-sided p-value for E[baseline loss - augmented loss] <= 0."""
    x = np.asarray(loss_diff, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 30:
        return np.nan

    observed = float(x.mean())
    centered = x - observed
    rng = np.random.default_rng(seed)
    boot_means = np.empty(replications, dtype=float)
    restart_prob = 1.0 / max(block_length, 1)

    for b in range(replications):
        sample = np.empty(n, dtype=float)
        idx = int(rng.integers(0, n))
        for t in range(n):
            if t == 0 or rng.random() < restart_prob:
                idx = int(rng.integers(0, n))
            else:
                idx = (idx + 1) % n
            sample[t] = centered[idx]
        boot_means[b] = sample.mean()

    return float((1.0 + np.sum(boot_means >= observed)) / (replications + 1.0))


def benjamini_hochberg(p_values: pd.Series) -> pd.Series:
    p = pd.to_numeric(p_values, errors="coerce")
    valid = p.dropna().sort_values()
    adjusted = pd.Series(np.nan, index=p.index, dtype=float)
    m = len(valid)
    if m == 0:
        return adjusted

    ranks = np.arange(1, m + 1)
    raw = valid.to_numpy() * m / ranks
    monotone = np.minimum.accumulate(raw[::-1])[::-1]
    adjusted.loc[valid.index] = np.clip(monotone, 0.0, 1.0)
    return adjusted


def summarize_forecasts(
    forecasts: pd.DataFrame,
    cluster: int,
    horizon: int,
) -> dict:
    y = forecasts["actual_log_variance"].to_numpy()
    b = forecasts["baseline_forecast"].to_numpy()
    a = forecasts["augmented_forecast"].to_numpy()

    b_error = y - b
    a_error = y - a
    b_qlike = qlike_loss(y, b)
    a_qlike = qlike_loss(y, a)
    loss_diff = b_qlike - a_qlike  # positive means the cluster model improves

    dm_stat, dm_p = newey_west_mean_test(loss_diff, CFG.dm_hac_lags)
    bootstrap_p = stationary_block_bootstrap_pvalue(
        loss_diff,
        CFG.bootstrap_replications,
        CFG.bootstrap_block_length,
        CFG.random_state + 1000 * horizon + cluster,
    )

    denominator = np.sum(np.square(y - np.mean(y)))
    oos_r2 = 1.0 - np.sum(np.square(a_error)) / denominator if denominator > 0 else np.nan
    baseline_oos_r2 = (
        1.0 - np.sum(np.square(b_error)) / denominator if denominator > 0 else np.nan
    )

    return {
        "cluster": int(cluster),
        "horizon_days": int(horizon),
        "test_start": forecasts.index.min(),
        "test_end": forecasts.index.max(),
        "test_n": int(len(forecasts)),
        "baseline_qlike": float(np.mean(b_qlike)),
        "augmented_qlike": float(np.mean(a_qlike)),
        "qlike_improvement": float(np.mean(loss_diff)),
        "qlike_improvement_pct": float(
            100.0 * (np.mean(b_qlike) - np.mean(a_qlike)) / abs(np.mean(b_qlike))
        ),
        "baseline_rmse_logvar": float(np.sqrt(np.mean(np.square(b_error)))),
        "augmented_rmse_logvar": float(np.sqrt(np.mean(np.square(a_error)))),
        "baseline_oos_r2": float(baseline_oos_r2),
        "augmented_oos_r2": float(oos_r2),
        "incremental_oos_r2": float(oos_r2 - baseline_oos_r2),
        "dm_hac_stat": dm_stat,
        "dm_hac_pvalue": dm_p,
        "block_bootstrap_pvalue": bootstrap_p,
    }


# -----------------------------------------------------------------------------
# Main execution
# -----------------------------------------------------------------------------

def run_step8() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    quality = prepare_term_quality_table()
    cluster_indices, index_audit = construct_cluster_indices(panel_norm, quality)
    market = load_market_variance(CFG.market_path)

    discovery_end = pd.Timestamp(CFG.discovery_end_date)
    if CFG.evaluation_mode == "strict_post_discovery":
        post_discovery_dates = market.index[market.index > discovery_end]
        if len(post_discovery_dates) < 60:
            raise ValueError(
                "Strict post-discovery evaluation requires at least 60 market "
                f"observations after {discovery_end.date()}, but found "
                f"{len(post_discovery_dates)}. Extend both market and Google Trends "
                "data without changing the frozen cluster definitions."
            )

    quality.to_csv(CFG.output_dir / "term_quality_and_weights.csv", index=False)
    cluster_indices.to_csv(CFG.output_dir / "cluster_attention_indices.csv")
    index_audit.to_csv(CFG.output_dir / "cluster_index_audit.csv", index=False)

    all_summaries = []
    forecast_tables = []

    for horizon in CFG.forecast_horizons:
        dataset = build_har_dataset(market, cluster_indices, horizon)

        if CFG.evaluation_mode == "strict_post_discovery":
            pre_holdout = dataset.loc[dataset.index <= discovery_end]
            first_cluster = cluster_indices.columns[0]
            initial_train_n = int(
                pre_holdout.dropna(
                    subset=["target", "har_d", "har_w", "har_m", first_cluster]
                ).shape[0]
            )
            if initial_train_n < CFG.min_initial_train_days:
                raise ValueError(
                    f"Only {initial_train_n} pre-discovery training rows are available."
                )
        else:
            common_n = dataset.dropna().shape[0]
            initial_train_n = max(
                CFG.min_initial_train_days,
                int(math.floor(common_n * CFG.initial_train_fraction)),
            )

        for index_name in cluster_indices.columns:
            cluster = int(index_name.split("_")[1])
            forecasts = expanding_forecasts(
                data=dataset,
                cluster_predictor=index_name,
                initial_train_n=initial_train_n,
                refit_every=CFG.refit_every,
            )

            if CFG.evaluation_mode == "strict_post_discovery":
                forecasts = forecasts.loc[forecasts.index > discovery_end]
                if forecasts.empty:
                    raise RuntimeError(
                        f"No strict holdout forecasts produced for cluster {cluster}."
                    )

            forecasts = forecasts.assign(
                cluster=cluster,
                horizon_days=horizon,
                evaluation_mode=CFG.evaluation_mode,
            )
            forecast_tables.append(forecasts.reset_index())
            all_summaries.append(summarize_forecasts(forecasts, cluster, horizon))

    results = pd.DataFrame(all_summaries)
    results["dm_hac_qvalue"] = (
        results.groupby("horizon_days", group_keys=False)["dm_hac_pvalue"]
        .apply(benjamini_hochberg)
    )
    results["block_bootstrap_qvalue"] = (
        results.groupby("horizon_days", group_keys=False)["block_bootstrap_pvalue"]
        .apply(benjamini_hochberg)
    )
    results["significant_5pct_fdr"] = (
        (results["qlike_improvement"] > 0)
        & (results["block_bootstrap_qvalue"] < 0.05)
    )
    results["evaluation_mode"] = CFG.evaluation_mode
    results["variance_proxy"] = market.attrs.get("variance_proxy", "unknown")

    forecasts_all = pd.concat(forecast_tables, ignore_index=True)
    results = results.sort_values(
        ["horizon_days", "augmented_qlike", "cluster"]
    ).reset_index(drop=True)

    results.to_csv(CFG.output_dir / "cluster_by_cluster_forecast_results.csv", index=False)
    forecasts_all.to_csv(CFG.output_dir / "all_oos_forecasts.csv", index=False)

    with open(CFG.output_dir / "step8_config.json", "w", encoding="utf-8") as handle:
        config_dict = asdict(CFG)
        config_dict["market_path"] = str(config_dict["market_path"])
        config_dict["output_dir"] = str(config_dict["output_dir"])
        json.dump(config_dict, handle, indent=2)

    # Publication-style summary plot: QLIKE gain with FDR-significance marker.
    for horizon, group in results.groupby("horizon_days"):
        plot_data = group.sort_values("qlike_improvement")
        fig, ax = plt.subplots(figsize=(8.5, 5.5))
        ax.barh(
            plot_data["cluster"].astype(str),
            plot_data["qlike_improvement"],
        )
        significant = plot_data["significant_5pct_fdr"].to_numpy()
        ax.scatter(
            plot_data.loc[significant, "qlike_improvement"],
            plot_data.loc[significant, "cluster"].astype(str),
            marker="*",
            s=90,
            label="FDR q < 0.05",
        )
        ax.axvline(0.0, linewidth=1.0)
        ax.set_title(f"Incremental QLIKE improvement by cluster: {horizon}-day horizon")
        ax.set_xlabel("Baseline QLIKE minus HAR-X QLIKE (higher is better)")
        ax.set_ylabel("Cluster")
        if significant.any():
            ax.legend(frameon=False)
        fig.tight_layout()
        fig.savefig(
            CFG.output_dir / f"qlike_improvement_h{horizon}.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()

    print("Step 8 output directory:", CFG.output_dir.resolve())
    print("Variance proxy:", market.attrs.get("variance_proxy"))
    print("Evaluation mode:", CFG.evaluation_mode)
    print(results.to_string(index=False))

    return results, cluster_indices, index_audit


step8_results, cluster_attention_indices, cluster_index_audit = run_step8()

Step 8 output directory: C:\Python\CSUREMM\output\08_prediction
Variance proxy: squared_log_return
Evaluation mode: exploratory_pseudo_oos
 cluster  horizon_days test_start   test_end  test_n  baseline_qlike  augmented_qlike  qlike_improvement  qlike_improvement_pct  baseline_rmse_logvar  augmented_rmse_logvar  baseline_oos_r2  augmented_oos_r2  incremental_oos_r2  dm_hac_stat  dm_hac_pvalue  block_bootstrap_pvalue  dm_hac_qvalue  block_bootstrap_qvalue  significant_5pct_fdr        evaluation_mode     variance_proxy
       9             1 2025-02-11 2026-05-28     325       -4.979646        -6.323293           1.343647              26.982784              2.486208               2.443091        -0.012271          0.022535            0.034806     1.322063       0.186147                0.099950       0.352324                0.290855                 False exploratory_pseudo_oos squared_log_return
       5             1 2025-02-11 2026-05-28     325       -4.979646        -5.095202          